# GPT-J Function Vector - Kaggle (produce-only, self-verifying)

Reproduces Todd et al. (2023) function-vector extraction + injection on **EleutherAI/gpt-j-6B**
using **nnsight** (with a baukit cross-check), ported from the repo's GPT-2 notebook.

**MANUAL TOGGLES (cannot be scripted - set these in the Kaggle right-hand panel):**
1. **Settings -> Internet: ON**  (needed for pip + weight download)
2. **Settings -> Accelerator: GPU**  (any 16 GB card, e.g. P100/T4)

Run the cells in order. Cell 2 loads GPT-J and **halts** so you can confirm the probed
shapes before extraction. Cell 3 asserts baukit==nnsight before building the FV.


In [1]:
# --- Cell 1: setup ---
import os
# HF_HOME must be set BEFORE importing transformers/nnsight. /kaggle/working has room
# for the ~12 GB fp16 GPT-J weights (default ~/.cache can be tight).
os.environ["HF_HOME"] = "/kaggle/working/hf"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Pin the stack this port was verified against. Do NOT reinstall torch: Kaggle's
# preinstalled torch is the CUDA build; replacing it would drop GPU support.
# baukit is NOT on PyPI - install from GitHub (same source the FV repo uses).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "nnsight==0.7.0", "transformers==5.13.0",
                "git+https://github.com/davidbau/baukit@main#egg=baukit"], check=True)

import torch, transformers, nnsight
print("=" * 60)
print(f"nnsight      : {nnsight.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"torch        : {torch.__version__}   (must stay the Kaggle CUDA build)")
print(f"CUDA         : {torch.cuda.is_available()}  "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else '<< set Accelerator=GPU >>'}")
print("=" * 60)
assert torch.cuda.is_available(), "No GPU visible. Set Accelerator=GPU (right-hand panel) and restart."


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.5/272.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 4.6 MB/s eta 0:00:00


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


nnsight      : 0.7.0
transformers : 5.13.0
torch        : 2.10.0+cu128   (must stay the Kaggle CUDA build)
CUDA         : True  Tesla T4


In [2]:
# --- Cell 2a: load GPT-J (fp16, GPU) -- own cell, safe to re-run ---
try:
    del model; import gc, torch; gc.collect(); torch.cuda.empty_cache()
except NameError:
    pass
import torch
from nnsight import LanguageModel

# revision="float16" pulls the ~12 GB fp16 branch, not the 24 GB fp32 weights.
# transformers 5.13.0 uses `dtype=`; if a build rejects it, swap to torch_dtype=torch.float16.
model = LanguageModel(
    "EleutherAI/gpt-j-6B",
    revision="float16",
    # PORT: fp16 weights so GPT-J 6B fits a 16 GB T4; stored FV norms are therefore fp16,
    # while the 40.939 reference norm is fp32 (.float().norm()).
    dtype=torch.float16,
    device_map="cuda",
    dispatch=True,
)
tokenizer = model.tokenizer
p0 = next(model._model.parameters())
print("loaded. dtype:", p0.dtype, "device:", p0.device)


config.json:   0%|          | 0.00/836 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/12.1G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

[transformers] GPTJForCausalLM LOAD REPORT from: EleutherAI/gpt-j-6B
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...27}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...27}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded. dtype: torch.float16 device: cuda:0


In [3]:
# --- Cell 2b: probe GPT-J and HALT (re-runs without reloading weights) ---
import torch

# GPT-J: 28 layers, 16 heads, head_dim 256, hidden 4096. (GPT-2 was 12/.. and fused c_attn/c_proj.)
attn = model.transformer.h[0].attn
print("attn children:", list(attn._modules.keys()))   # expect q_proj/k_proj/v_proj/out_proj (no bias)
print("out_proj      :", attn.out_proj)

def describe(name, x):
    if isinstance(x, tuple):
        print(f"{name}: tuple(len={len(x)}) ->",
              [tuple(t.shape) if torch.is_tensor(t) else type(t).__name__ for t in x])
    else:
        print(f"{name}: {type(x).__name__} shape="
              f"{tuple(x.shape) if torch.is_tensor(x) else '?'}")

ids = tokenizer("Hello world", return_tensors="pt")["input_ids"].cuda()
with model.trace(ids):
    op_in  = model.transformer.h[0].attn.out_proj.input.save()   # per-head source (pre-projection)
    attn_o = model.transformer.h[0].attn.output.save()           # attn submodule output
    blk_o  = model.transformer.h[9].output.save()                # block output = injection point

print("\n--- probed forms (confirm before Cell 3) ---")
describe("attn.out_proj.input", op_in)   # verified on gpt2: a bare Tensor [1,S,4096]
describe("attn.output        ", attn_o)  # tuple -> [0] is the tensor
describe("h[9].output (block)", blk_o)   # GPT-J block returns a TUPLE -> [0] is hidden; inject here

N_HEADS, HEAD_DIM, HIDDEN = 16, 256, 4096
assert N_HEADS * HEAD_DIM == HIDDEN, (N_HEADS, HEAD_DIM, HIDDEN)
print(f"\nPer-head split : out_proj.input [1,S,{HIDDEN}] -> [1,S,{N_HEADS},{HEAD_DIM}]")
print("Injection      : model.transformer.h[9].output[0][:, -1, :] += fv_vector")
print("\n*** HALT: check attn.out_proj.input is a [1,S,4096] tensor and h[9].output[0] is [1,S,4096]. "
      "Then run Cell 3. ***")


attn children: ['attn_dropout', 'resid_dropout', 'k_proj', 'v_proj', 'q_proj', 'out_proj']
out_proj      : Linear(in_features=4096, out_features=4096, bias=False)

--- probed forms (confirm before Cell 3) ---
attn.out_proj.input: Tensor shape=(1, 2, 4096)
attn.output        : tuple(len=2) -> [(1, 2, 4096), (1, 16, 2, 2)]
h[9].output (block): tuple(len=2) -> [(1, 2, 4096), (1, 16, 2, 2)]

Per-head split : out_proj.input [1,S,4096] -> [1,S,16,256]
Injection      : model.transformer.h[9].output[0][:, -1, :] += fv_vector

*** HALT: check attn.out_proj.input is a [1,S,4096] tensor and h[9].output[0] is [1,S,4096]. Then run Cell 3. ***


In [4]:
# --- Cell 3: baukit<->nnsight match, then build FV and inject ---
import json, random
import numpy as np
import torch
from baukit import TraceDict

# ---- config (GPT-J; corrected from the GPT-2 port: 16 heads x 256, NOT 25 x 164) ----
N_HEADS, HEAD_DIM, RESID_DIM, N_LAYERS = 16, 256, 4096, 28
# PORT: EDIT_LAYER=9 ~= one third of GPT-J's 28 layers (Todd); exact match to his reported layer.
EDIT_LAYER = 9                      # ~= |L|/3, the paper's GPT-J intervention layer
N_ICL, N_TRIALS, N_TEST, SEED = 10, 100, 40, 0
# 10 universal high-AIE heads for GPT-J (Todd et al.); head indices are valid for 16 heads.
# PORT/DEVIATION: Todd's 10 published high-AIE heads reused VERBATIM. Causal mediation was NOT
# rerun -- this is a deviation from a full reproduction, not merely a detail.
UNIVERSAL_TOP_INDICES = [(15,5),(9,14),(12,10),(8,1),(11,0),(13,13),(8,0),(14,9),(9,2),(24,6)]

# country-capital pairs embedded (ASCII-safe JSON; == function_vectors/.../country-capital.json)
PAIRS = [tuple(x) for x in json.loads(r"""[["Afghanistan", "Kabul"], ["Albania", "Tirana"], ["Algeria", "Algiers"], ["Andorra", "Andorra la Vella"], ["Angola", "Luanda"], ["Antigua and Barbuda", "St. John's"], ["Argentina", "Buenos Aires"], ["Armenia", "Yerevan"], ["Australia", "Canberra"], ["Austria", "Vienna"], ["Azerbaijan", "Baku"], ["Bahamas", "Nassau"], ["Bahrain", "Manama"], ["Bangladesh", "Dhaka"], ["Barbados", "Bridgetown"], ["Belarus", "Minsk"], ["Belgium", "Brussels"], ["Belize", "Belmopan"], ["Benin", "Porto-Novo"], ["Bhutan", "Thimphu"], ["Bolivia", "La Paz"], ["Bosnia and Herzegovina", "Sarajevo"], ["Botswana", "Gaborone"], ["Brazil", "Brasilia"], ["Brunei", "Bandar Seri Begawan"], ["Bulgaria", "Sofia"], ["Burkina Faso", "Ouagadougou"], ["Burundi", "Bujumbura"], ["Cabo Verde", "Praia"], ["Cambodia", "Phnom Penh"], ["Cameroon", "Yaounde"], ["Canada", "Ottawa"], ["Central African Republic", "Bangui"], ["Chad", "N'Djamena"], ["Chile", "Santiago"], ["China", "Beijing"], ["Colombia", "Bogot\u00e1"], ["Comoros", "Moroni"], ["Congo", "Kinshasa"], ["Costa Rica", "San Jos\u00e9"], ["Cote d'Ivoire", "Yamoussoukro"], ["Croatia", "Zagreb"], ["Cuba", "Havana"], ["Cyprus", "Nicosia"], ["Czech Republic", "Prague"], ["Democratic Republic of the Congo", "Kinshasa"], ["Denmark", "Copenhagen"], ["Djibouti", "Djibouti City"], ["Dominica", "Roseau"], ["Dominican Republic", "Santo Domingo"], ["Ecuador", "Quito"], ["Egypt", "Cairo"], ["El Salvador", "San Salvador"], ["Equatorial Guinea", "Malabo"], ["Eritrea", "Asmara"], ["Estonia", "Tallinn"], ["Eswatini", "Mbabane"], ["Ethiopia", "Addis Ababa"], ["Fiji", "Suva"], ["Finland", "Helsinki"], ["France", "Paris"], ["Gabon", "Libreville"], ["Gambia", "Banjul"], ["Georgia", "Tbilisi"], ["Germany", "Berlin"], ["Ghana", "Accra"], ["Greece", "Athens"], ["Grenada", "St. George's"], ["Guatemala", "Guatemala City"], ["Guinea", "Conakry"], ["Guinea-Bissau", "Bissau"], ["Guyana", "Georgetown"], ["Haiti", "Port-au-Prince"], ["Honduras", "Tegucigalpa"], ["Hungary", "Budapest"], ["Iceland", "Reykjavik"], ["India", "New Delhi"], ["Indonesia", "Jakarta"], ["Iran", "Tehran"], ["Iraq", "Baghdad"], ["Ireland", "Dublin"], ["Israel", "Jerusalem"], ["Italy", "Rome"], ["Jamaica", "Kingston"], ["Japan", "Tokyo"], ["Jordan", "Amman"], ["Kazakhstan", "Astana"], ["Kenya", "Nairobi"], ["Kiribati", "South Tarawa"], ["Kosovo", "Pristina"], ["Kuwait", "Kuwait City"], ["Kyrgyzstan", "Bishkek"], ["Laos", "Vientiane"], ["Latvia", "Riga"], ["Lebanon", "Beirut"], ["Lesotho", "Maseru"], ["Liberia", "Monrovia"], ["Libya", "Tripoli"], ["Liechtenstein", "Vaduz"], ["Lithuania", "Vilnius"], ["Luxembourg", "Luxembourg City"], ["Madagascar", "Antananarivo"], ["Malawi", "Lilongwe"], ["Malaysia", "Kuala Lumpur"], ["Maldives", "Mal\u00e9"], ["Mali", "Bamako"], ["Malta", "Valletta"], ["Marshall Islands", "Majuro"], ["Mauritania", "Nouakchott"], ["Mauritius", "Port Louis"], ["Mexico", "Mexico City"], ["Micronesia", "Palikir"], ["Moldova", "Chisinau"], ["Monaco", "Monaco-Ville"], ["Mongolia", "Ulaanbaatar"], ["Montenegro", "Podgorica"], ["Morocco", "Rabat"], ["Mozambique", "Maputo"], ["Myanmar", "Naypyidaw"], ["Namibia", "Windhoek"], ["Nauru", "Yaren District"], ["Nepal", "Kathmandu"], ["Netherlands", "Amsterdam"], ["New Zealand", "Wellington"], ["Nicaragua", "Managua"], ["Niger", "Niamey"], ["Nigeria", "Abuja"], ["North Korea", "Pyongyang"], ["North Macedonia", "Skopje"], ["Norway", "Oslo"], ["Oman", "Muscat"], ["Pakistan", "Islamabad"], ["Palau", "Ngerulmud"], ["Palestine", "Ramallah"], ["Panama", "Panama City"], ["Papua New Guinea", "Port Moresby"], ["Paraguay", "Asunci\u00f3n"], ["Peru", "Lima"], ["Philippines", "Manila"], ["Poland", "Warsaw"], ["Portugal", "Lisbon"], ["Qatar", "Doha"], ["Romania", "Bucharest"], ["Russia", "Moscow"], ["Rwanda", "Kigali"], ["Saint Kitts and Nevis", "Basseterre"], ["Saint Lucia", "Castries"], ["Saint Vincent and the Grenadines", "Kingstown"], ["Samoa", "Apia"], ["San Marino", "San Marino"], ["Sao Tome and Principe", "Sao Tome"], ["Saudi Arabia", "Riyadh"], ["Senegal", "Dakar"], ["Serbia", "Belgrade"], ["Seychelles", "Victoria"], ["Sierra Leone", "Freetown"], ["Singapore", "Singapore"], ["Slovakia", "Bratislava"], ["Slovenia", "Ljubljana"], ["Solomon Islands", "Honiara"], ["Somalia", "Mogadishu"], ["South Africa", "Pretoria"], ["South Korea", "Seoul"], ["South Sudan", "Juba"], ["Spain", "Madrid"], ["Sri Lanka", "Colombo"], ["Sudan", "Khartoum"], ["Suriname", "Paramaribo"], ["Sweden", "Stockholm"], ["Switzerland", "Bern"], ["Syria", "Damascus"], ["Taiwan", "Taipei"], ["Tajikistan", "Dushanbe"], ["Tanzania", "Dodoma"], ["Thailand", "Bangkok"], ["Timor-Leste", "Dili"], ["Togo", "Lome"], ["Tonga", "Nuku\u02bbalofa"], ["Trinidad and Tobago", "Port of Spain"], ["Tunisia", "Tunis"], ["Turkey", "Ankara"], ["Turkmenistan", "Ashgabat"], ["Tuvalu", "Funafuti"], ["Uganda", "Kampala"], ["Ukraine", "Kiev"], ["United Arab Emirates", "Abu Dhabi"], ["United Kingdom", "London"], ["United States of America", "Washington, D.C."], ["Uruguay", "Montevideo"], ["Uzbekistan", "Tashkent"], ["Vanuatu", "Port Vila"], ["Vatican City", "Vatican City"], ["Venezuela", "Caracas"], ["Vietnam", "Hanoi"], ["Yemen", "Sana'a"], ["Zambia", "Lusaka"], ["Zimbabwe", "Harare"]]""")]
print(f"{len(PAIRS)} country-capital pairs")

def as_tensor(x):
    """out_proj.input is a bare tensor here, but unwrap (args,kwargs) forms defensively."""
    while isinstance(x, (tuple, list)):
        x = x[0]
    return x

# PORT: local random.Random(seed) -- split is deterministic per seed, independent of global RNG.
def split_data(pairs, n_test=N_TEST, seed=SEED):
    rng = random.Random(seed)
    idx = list(range(len(pairs))); rng.shuffle(idx)
    test = [pairs[i] for i in idx[:n_test]]
    pool = [pairs[i] for i in idx[n_test:]]
    return pool, test

def make_icl_prompt(pool, n_icl, rng):
    """N_ICL Todd et al. Q/A demos joined by blank lines, then the held-out query as
    'Q: {country}\nA:' with NO capital and NO trailing space, so the prompt ends on the
    answer-slot marker (a separate position from the query country token) where the FV
    is injected and the next-token gold stays space-prefixed."""
    sample = rng.sample(pool, n_icl + 1)                # +1 distinct item = the query (no leakage)
    demos = [f"Q: {c}\nA: {cap}" for c, cap in sample[:n_icl]]
    query = f"Q: {sample[n_icl][0]}\nA:"                # query country, no capital, no trailing space
    return "\n\n".join(demos + [query])

ICL_POOL, TEST = split_data(PAIRS)
assert not (set(ICL_POOL) & set(TEST)), "held-out test leaked into ICL pool"
print(f"ICL pool={len(ICL_POOL)}  held-out test={len(TEST)}  (disjoint)")

# PORT: hooking via nnsight, not baukit (Todd's reference impl). This block cross-checks the
# two on out_proj input; the assert requires max|diff| < 1e-4 (project record: observed 0.000e+00).
# ---- 1) baukit vs nnsight: same weights + input must give the same out_proj.input ----
probe_ids = tokenizer(make_icl_prompt(ICL_POOL, N_ICL, random.Random(SEED)),
                      return_tensors="pt")["input_ids"].cuda()
import re
names = [f"transformer.h.{L}.attn.out_proj" for L in range(N_LAYERS)]
# Derive the layer list FROM `names` so the nnsight and baukit sides can't drift.
# Ascending order = forward-pass order, so nnsight can't deadlock.
layer_of = lambda n: int(re.search(r"\.h\.(\d+)\.", n).group(1))
uniq_layers = sorted({layer_of(n) for n in names})
print("names[0]:", names[0], "| len(names):", len(names))

# nnsight 0.7.0: a comprehension assigned INSIDE a trace does NOT bind after exit
# (verified on gpt2: UnboundLocalError). Pre-define the dict, fill via a plain loop.
ns_by_layer = {}
with model.trace(probe_ids):
    for L in uniq_layers:
        ns_by_layer[L] = model.transformer.h[L].attn.out_proj.input.save()

with torch.no_grad(), TraceDict(model._model, names, retain_input=True, retain_output=False) as td:
    model._model(probe_ids)

max_diff = 0.0
for name in names:
    a = as_tensor(ns_by_layer[layer_of(name)]).float()
    b = as_tensor(td[name].input).float()
    max_diff = max(max_diff, (a - b).abs().max().item())
print(f"baukit vs nnsight  max|out_proj.input diff| = {max_diff:.3e}")
assert max_diff < 1e-4, f"hooking methods disagree ({max_diff}); do not proceed"
print("OK: hooking methods agree -> extract with nnsight")

# ---- 2) mean head activations at the last token, over N_TRIALS ICL prompts ----
def collect_mean_head_activations(pool, n_trials=N_TRIALS, n_icl=N_ICL, seed=SEED):
    rng = random.Random(seed)
    acc = torch.zeros(N_LAYERS, N_HEADS, HEAD_DIM, dtype=torch.float32)   # CPU running sum
    for _ in range(n_trials):
        ids = tokenizer(make_icl_prompt(pool, n_icl, rng), return_tensors="pt")["input_ids"].cuda()
        saved = []
        # no_grad drops the autograd graph the saved proxies would retain (the OOM cause);
        # saving only the last token caps retained activation to one forward pass.
        with torch.no_grad(), model.trace(ids):
            for L in range(N_LAYERS):
                saved.append(model.transformer.h[L].attn.out_proj.input[:, -1, :].save())  # [1, 4096]
        for L in range(N_LAYERS):
            last = as_tensor(saved[L]).cpu().float()      # to CPU float32 immediately
            acc[L] += last.view(N_HEADS, HEAD_DIM)        # same 16x256 split, on the last token
    return (acc / n_trials).to("cuda")                    # [L, H, D] cuda float32 = unchanged contract

# ---- 3) FV = sum over top heads of out_proj(head slotted into its slice) ----
def compute_function_vector(mean_act, top_indices=UNIVERSAL_TOP_INDICES):
    fv = torch.zeros(1, RESID_DIM, device="cuda", dtype=torch.float16)
    for L, H in top_indices:
        out_proj = model.transformer.h[L].attn.out_proj
        slot = torch.zeros(RESID_DIM, device="cuda", dtype=out_proj.weight.dtype)
        slot[H*HEAD_DIM:(H+1)*HEAD_DIM] = mean_act[L, H].to(out_proj.weight.dtype)
        fv = fv + out_proj(slot.unsqueeze(0))
    return fv

mean_act = collect_mean_head_activations(ICL_POOL)
fv_vector = compute_function_vector(mean_act)
print("mean_act:", tuple(mean_act.shape), "| fv_vector:", tuple(fv_vector.shape),
      "| norm:", round(fv_vector.float().norm().item(), 3))

# ---- 4) zero-shot capital-naming accuracy, with vs without FV at layer 9 ----
# PORT: gold id = leading-space first BPE token via tokenizer(" " + word) -- grades the
# space-prefixed token emitted after the answer slot.
def first_token_id(capital):
    # in-context capitals are space-prefixed; grade on the space-prefixed first BPE token
    return tokenizer(" " + capital, add_special_tokens=False)["input_ids"][0]

# PORT: Q:\nA: framing. framed() / make_icl_prompt() are task-agnostic -- the SAME framing is
# applied uniformly across all three tasks (country_capital, antonym, present_past) in Cell 9.
def framed(country):
    # Todd et al. query form, identical to make_icl_prompt's held-out query:
    # 'Q: {country}\nA:' with no answer and no trailing space, so the FV lands on the
    # 'A:' answer-slot marker and the model predicts the leading-space capital (= gold).
    return f"Q: {country}\nA:"

def clean_pred(country):
    ids = tokenizer(framed(country), return_tensors="pt")["input_ids"].cuda()
    with model.trace(ids):
        logits = model.output.logits.save()
    return int(logits[0, -1].argmax())

def fv_pred(country):
    ids = tokenizer(framed(country), return_tensors="pt")["input_ids"].cuda()
    with model.trace(ids):
        h = model.transformer.h[EDIT_LAYER].output[0]     # GPT-J block output is a tuple -> [0]
        # PORT: injection site -- additive at the final position of block EDIT_LAYER's output[0].
        h[:, -1, :] = h[:, -1, :] + fv_vector
        logits = model.output.logits.save()
    return int(logits[0, -1].argmax())

# ---- PROOF: framed eval query mirrors make_icl_prompt's per-query format ----
_pc, _pcap = TEST[0]
_fq = framed(_pc)
_mi = make_icl_prompt(ICL_POOL, N_ICL, random.Random(SEED))
print("framed eval query (first test country):", repr(_fq))
print("make_icl_prompt sample tail            :", repr(_mi[-40:]))
print("framed query last-3 ids                :", tokenizer(_fq)["input_ids"][-3:])
print("make_icl_prompt last-3 ids             :", tokenizer(_mi)["input_ids"][-3:])

# ---- DIAGNOSTIC (added): raw output; reuses clean_pred/fv_pred AS-IS, no accuracy ----
print("token check:")
for w in ["Paris", " Paris"]:
    print(" ", repr(w), tokenizer(w)["input_ids"])
_dc, _dcap = TEST[0]
print("harness gold id for", repr(_dcap), "=", first_token_id(_dcap),
      "| full ids:", tokenizer(" " + _dcap, add_special_tokens=False)["input_ids"])
print("\nper-country raw top-1 (first 5 held-out; no accuracy):")
for country, capital in TEST[:5]:
    gold = first_token_id(capital)
    gold_ids = tokenizer(" " + capital, add_special_tokens=False)["input_ids"]
    cid = clean_pred(country)            # no FV
    fid = fv_pred(country)               # WITH FV, injection exactly as written
    print(f"  {country!r:30} gold={capital!r} gold_id={gold} gold_ids={gold_ids} "
          f"| noFV: id={cid} {tokenizer.decode([cid])!r} "
          f"| FV: id={fid} {tokenizer.decode([fid])!r}")

clean_hits, fv_hits = 0, 0
# PORT: denominator = ALL held-out queries. Todd instead filters to queries the model answers
# correctly under 10-shot ICL, so these numbers are stricter and not directly comparable to his.
for country, capital in TEST:
    gold = first_token_id(capital)
    clean_hits += (clean_pred(country) == gold)
    fv_hits    += (fv_pred(country) == gold)

n = len(TEST)
clean_acc, fv_acc = clean_hits / n, fv_hits / n
print(f"\nZero-shot capital naming on {n} held-out countries (first-token match):")
print(f"  without FV : {clean_acc:.3f}  ({clean_hits}/{n})")
print(f"  with FV @L9: {fv_acc:.3f}  ({fv_hits}/{n})")
print(f"  delta      : {fv_acc - clean_acc:+.3f}")


197 country-capital pairs
ICL pool=157  held-out test=40  (disjoint)
names[0]: transformer.h.0.attn.out_proj | len(names): 28
baukit vs nnsight  max|out_proj.input diff| = 0.000e+00
OK: hooking methods agree -> extract with nnsight
mean_act: (28, 16, 256) | fv_vector: (1, 4096) | norm: 40.939
framed eval query (first test country): 'Q: Laos\nA:'
make_icl_prompt sample tail            : 'dor\n\nQ: Canada\nA: Ottawa\n\nQ: Malaysia\nA:'
framed query last-3 ids                : [198, 32, 25]
make_icl_prompt last-3 ids             : [198, 32, 25]
token check:
  'Paris' [40313]
  ' Paris' [6342]
harness gold id for 'Vientiane' = 569 | full ids: [569, 1153, 46470]

per-country raw top-1 (first 5 held-out; no accuracy):
  'Laos'                         gold='Vientiane' gold_id=569 gold_ids=[569, 1153, 46470] | noFV: id=45919 ' Laos' | FV: id=569 ' V'
  'Finland'                      gold='Helsinki' gold_id=47688 gold_ids=[47688] | noFV: id=17837 ' Finland' | FV: id=47688 ' Helsinki'
  'Taiwan'

In [5]:
# --- Cell 5 (DIAGNOSTIC, added -- new cell; no existing cell modified) ---
# For the first 10 held-out TEST countries, run framed(country) with no injection and
# with scale*fv_vector for scale in {1,3,5,10}, each in its own trace. Report the gold
# first-token logprob and rank (noFV and FV 1x) and the top-1 token at each scale.
# Injection formula copied verbatim from Cell 3's fv_pred (only `scale *` added on
# fv_vector); gold id from Cell 3's first_token_id, unchanged.
import torch

def _clean_logits(country):                                   # trace body == Cell 3 clean_pred
    ids = tokenizer(framed(country), return_tensors="pt")["input_ids"].cuda()
    with model.trace(ids):
        logits = model.output.logits.save()
    return logits[0, -1].float().cpu()                        # final-token logit vector [vocab]

def _fv_logits(country, scale):                               # trace body == Cell 3 fv_pred + `scale *`
    ids = tokenizer(framed(country), return_tensors="pt")["input_ids"].cuda()
    with model.trace(ids):
        h = model.transformer.h[EDIT_LAYER].output[0]         # GPT-J block output is a tuple -> [0]
        h[:, -1, :] = h[:, -1, :] + scale * fv_vector
        logits = model.output.logits.save()
    return logits[0, -1].float().cpu()

def _lp_rank(vec, gold):
    lp = torch.log_softmax(vec, dim=-1)
    rank = int((vec > vec[gold]).sum().item()) + 1            # 1-indexed: rank 1 == argmax
    return lp[gold].item(), rank

def _top1(vec):
    return tokenizer.decode([int(vec.argmax())])

SCALES = [1, 3, 5, 10]                                        # 1x == Cell 3 unscaled inject (1*fv)
print("DIAGNOSTIC: first 10 held-out TEST countries")
print(f"{'country':22} {'gold':>10} |   noFV lp / rank |   FV1x lp / rank | top1  1x / 3x / 5x / 10x")
print("-" * 120)
for country, capital in TEST[:10]:
    gold  = first_token_id(capital)                           # Cell 3 gold logic, unchanged
    gstr  = tokenizer.decode([gold])
    clean = _clean_logits(country)
    fvs   = {s: _fv_logits(country, s) for s in SCALES}
    c_lp, c_rk = _lp_rank(clean, gold)
    f_lp, f_rk = _lp_rank(fvs[1], gold)
    tops = " / ".join(repr(_top1(fvs[s])) for s in SCALES)
    print(f"{country[:22]:22} {gstr!r:>10} | {c_lp:8.3f} / {c_rk:<6} | {f_lp:8.3f} / {f_rk:<6} | {tops}")

# ---- read-back (magnitude-independent): did the EDIT_LAYER last-token slice actually change? ----
# Independent of logits and of GPT-J's large residual norms. The += below is the EXACT line
# from Cell 3's fv_pred (1x, no scale); only .clone()/.save() are added to snapshot the slice
# AFTER the mutation. Interpretation left to the reader (nothing stated here).
def _readback_diff(country):
    ids = tokenizer(framed(country), return_tensors="pt")["input_ids"].cuda()
    with model.trace(ids):                                    # trace 1: NO injection
        clean = model.transformer.h[EDIT_LAYER].output[0][:, -1, :].clone().save()
    with model.trace(ids):                                    # trace 2: injection, read back AFTER
        h = model.transformer.h[EDIT_LAYER].output[0]         # GPT-J block output is a tuple -> [0]
        h[:, -1, :] = h[:, -1, :] + fv_vector                 # Cell 3 fv_pred += line, verbatim
        injected = h[:, -1, :].clone().save()                 # slice AFTER the mutation
    return (injected - clean).float().norm().item()

fv_norm = fv_vector.float().norm().item()
print("\nread-back: ||injected - clean||_2 of EDIT_LAYER last-token slice vs ||fv_vector||_2")
print(f"{'country':22} | {'||inj-clean||':>13} | {'||fv_vector||':>13}")
print("-" * 56)
for country, _ in TEST[:10]:
    print(f"{country[:22]:22} | {_readback_diff(country):13.4f} | {fv_norm:13.4f}")

DIAGNOSTIC: first 10 held-out TEST countries
country                      gold |   noFV lp / rank |   FV1x lp / rank | top1  1x / 3x / 5x / 10x
------------------------------------------------------------------------------------------------------------------------
Laos                         ' V' |   -5.640 / 32     |   -0.749 / 1      | ' V' / ' V' / ' V' / ' Lu'
Finland                ' Helsinki' |   -6.501 / 88     |   -1.243 / 1      | ' Helsinki' / ' Helsinki' / ' Helsinki' / ' Helsinki'
Taiwan                     ' Tai' |   -5.456 / 25     |   -0.786 / 1      | ' Tai' / ' Tai' / ' Tai' / ' Tai'
Peru                      ' Lima' |   -6.331 / 71     |   -0.597 / 1      | ' Lima' / ' Lima' / ' Lima' / ' Lima'
Fiji                        ' Su' |   -8.151 / 345    |   -4.961 / 13     | ' Fiji' / ' Su' / ' Su' / ' Man'
Morocco                    ' Rab' |   -8.076 / 369    |   -2.326 / 3      | ' Mar' / ' Mar' / ' Mar' / ' Al'
Philippines             ' Manila' |   -6.014 / 46     |   -

In [6]:
# --- Cell 4: save artifacts to /kaggle/working ---
import json, torch
torch.save(fv_vector.cpu(), "/kaggle/working/gptj_country_capital_fv.pt")
with open("/kaggle/working/fv_results.json", "w") as f:
    json.dump({
        "model": "EleutherAI/gpt-j-6B",
        "edit_layer": EDIT_LAYER,
        "top_heads": UNIVERSAL_TOP_INDICES,
        "baukit_nnsight_max_diff": max_diff,
        "zeroshot_clean_acc": clean_acc,
        "zeroshot_fv_acc": fv_acc,
        "zeroshot_delta": fv_acc - clean_acc,
        "n_test": len(TEST), "n_icl": N_ICL, "n_trials": N_TRIALS,
    }, f, indent=2)
print("saved: /kaggle/working/gptj_country_capital_fv.pt, /kaggle/working/fv_results.json")


saved: /kaggle/working/gptj_country_capital_fv.pt, /kaggle/working/fv_results.json


In [7]:
# --- Cell 6 (added -- 3-seed first-token eval; no existing cell modified) ---
# Repeats Cell 3's single-seed extraction+eval across SEEDS, reusing existing functions
# unchanged. fv_vector is reassigned at module scope each seed so the existing fv_pred
# (which reads fv_vector as a global) picks it up. Model load stays in Cell 2a -- NOT
# reloaded here (reloading GPT-J is the OOM path). First-token metric only.
import torch, random
import numpy as np

SEEDS = [0, 1, 2]
base_accs, fv_accs = [], []
for s in SEEDS:
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    with torch.no_grad():
        ICL_POOL, TEST = split_data(PAIRS, seed=s)                   # rebuild split from SEED
        mean_act  = collect_mean_head_activations(ICL_POOL, seed=s)  # existing fn, unchanged
        fv_vector = compute_function_vector(mean_act)                # module-scope reassign -> fv_pred sees it
        n = len(TEST)
        base = sum(clean_pred(c) == first_token_id(cap) for c, cap in TEST) / n
        fv   = sum(fv_pred(c)    == first_token_id(cap) for c, cap in TEST) / n
    torch.cuda.empty_cache()
    base_accs.append(base); fv_accs.append(fv)
    print(f"seed {s}: baseline={base:.3f}  FV@L{EDIT_LAYER}={fv:.3f}  (n={n}, TEST[0]={TEST[0][0]!r})")

print(f"\nacross {len(SEEDS)} seeds (first-token match):")
print(f"  baseline: mean={np.mean(base_accs):.3f}  std={np.std(base_accs):.3f}")
print(f"  FV @L{EDIT_LAYER}: mean={np.mean(fv_accs):.3f}  std={np.std(fv_accs):.3f}")

seed 0: baseline=0.075  FV@L9=0.825  (n=40, TEST[0]='Laos')
seed 1: baseline=0.050  FV@L9=0.750  (n=40, TEST[0]='Bhutan')
seed 2: baseline=0.025  FV@L9=0.925  (n=40, TEST[0]='Bolivia')

across 3 seeds (first-token match):
  baseline: mean=0.050  std=0.020
  FV @L9: mean=0.833  std=0.072


In [8]:
# --- Cell 7 (added -- 5-seed first-token eval + FV cache; SUPERSEDES the 3-seed cell 9fca5b96) ---
# SUPERSEDES the earlier 3-seed cell (id 9fca5b96) -- that cell can be skipped.
# Same extraction+eval as Cell 3 / the 3-seed cell, reused unchanged, now over 5 seeds with
# SAMPLE std (ddof=1; population std understates n=5 spread). Also caches each seed's
# fv_vector + TEST in SEED_ARTIFACTS so the next cell re-extracts nothing. Model load stays
# in Cell 2a -- never reloaded here (reloading GPT-J is the OOM path). First-token metric only.
import torch, random
import numpy as np

SEEDS = [0, 1, 2, 3, 4]
SEED_ARTIFACTS = {}
base_accs, fv_accs = [], []
for s in SEEDS:
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    with torch.no_grad():
        ICL_POOL, TEST = split_data(PAIRS, seed=s)                   # rebuild split from SEED
        mean_act  = collect_mean_head_activations(ICL_POOL, seed=s)  # existing fn, unchanged
        fv_vector = compute_function_vector(mean_act)                # module-scope reassign -> fv_pred sees it
        n = len(TEST)
        base = sum(clean_pred(c) == first_token_id(cap) for c, cap in TEST) / n
        fv   = sum(fv_pred(c)    == first_token_id(cap) for c, cap in TEST) / n
    SEED_ARTIFACTS[s] = {"fv_vector": fv_vector, "TEST": TEST}       # cache for the exact-match cell
    torch.cuda.empty_cache()
    base_accs.append(base); fv_accs.append(fv)
    print(f"seed {s}: baseline={base:.3f}  FV@L{EDIT_LAYER}={fv:.3f}  (n={n}, TEST[0]={TEST[0][0]!r})")

print(f"\nacross {len(SEEDS)} seeds (first-token match, sample std ddof=1):")
print(f"  baseline: mean={np.mean(base_accs):.3f}  std={np.std(base_accs, ddof=1):.3f}")
print(f"  FV @L{EDIT_LAYER}: mean={np.mean(fv_accs):.3f}  std={np.std(fv_accs, ddof=1):.3f}")

seed 0: baseline=0.075  FV@L9=0.825  (n=40, TEST[0]='Laos')
seed 1: baseline=0.050  FV@L9=0.750  (n=40, TEST[0]='Bhutan')
seed 2: baseline=0.025  FV@L9=0.925  (n=40, TEST[0]='Bolivia')
seed 3: baseline=0.075  FV@L9=0.825  (n=40, TEST[0]='Palau')
seed 4: baseline=0.025  FV@L9=0.875  (n=40, TEST[0]='Micronesia')

across 5 seeds (first-token match, sample std ddof=1):
  baseline: mean=0.050  std=0.025
  FV @L9: mean=0.840  std=0.065


In [9]:
# --- Cell 8 (added -- exact-capital match, SELF-CONTAINED; re-extracts, does NOT read SEED_ARTIFACTS) ---
# Greedy free-generation exact-capital match. Re-extracts fv_vector per seed (same calls as
# the 5-seed cell); does NOT read SEED_ARTIFACTS. Reuses the SAME injection statement as
# dd1aa448's fv_pred, copied VERBATIM -- no new injection code, no model.generate().
# DECISION (option B): FV injected at the CURRENT last position at EVERY generation step,
# reproducing fv_pred's single-forward injection at step 0. NOT implemented (option A): inject
# once at the prompt boundary, generate the rest clean. If exact numbers look degenerate,
# flip this first.
# The verbatim generation loop lives in _fv_generate so exact_capital_match (-> bool) and the
# seed-0 display table share ONE injection path. Model load stays in Cell 2a -- never reloaded.
import torch, random
import numpy as np

# PORT: hand-rolled greedy generation (not model.generate) so FV injection is controlled per
# step. Known artifact: trailing junk on some multi-token outputs; option A (inject once at the
# prompt boundary, then generate clean) is the first knob to flip if outputs look degenerate.
def _fv_generate(country, k):
    # greedy k-token generation; injection body VERBATIM from dd1aa448.fv_pred, additions = append+loop
    ids = tokenizer(framed(country), return_tensors="pt")["input_ids"].cuda()
    generated = []
    with torch.no_grad():
        for _ in range(k):
            with model.trace(ids):
                h = model.transformer.h[EDIT_LAYER].output[0]     # GPT-J block output is a tuple -> [0]
                h[:, -1, :] = h[:, -1, :] + fv_vector
                logits = model.output.logits.save()
            nxt = int(logits[0, -1].argmax().cpu())               # argmax to CPU
            generated.append(nxt)
            ids = torch.cat([ids, torch.tensor([[nxt]], device=ids.device)], dim=1)
    return generated

# PORT: exact match = strict string equality, NO alias table. Near-misses (e.g. Tarawa vs
# South Tarawa) score as misses, which costs real points.
def exact_capital_match(country, capital):
    k = len(tokenizer(" " + capital, add_special_tokens=False)["input_ids"])
    gen = tokenizer.decode(_fv_generate(country, k)).strip().lower()
    return gen == capital.strip().lower()

SEEDS = [0, 1, 2, 3, 4]
exact_accs, ft_accs = [], []
seed0_rows = None
for s in SEEDS:
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    with torch.no_grad():
        ICL_POOL, TEST = split_data(PAIRS, seed=s)                   # same call increment 1 used
        mean_act  = collect_mean_head_activations(ICL_POOL, seed=s)  # existing fn, unchanged
        fv_vector = compute_function_vector(mean_act)                # module-scope reassign
        n = len(TEST)
        exact = sum(exact_capital_match(c, cap) for c, cap in TEST) / n
        ft    = sum(fv_pred(c) == first_token_id(cap) for c, cap in TEST) / n   # side cross-check vs increment 1
        if s == 0:                                                   # capture generated strings for the table
            seed0_rows = []
            for c, cap in TEST:
                kk = len(tokenizer(" " + cap, add_special_tokens=False)["input_ids"])
                seed0_rows.append((c, cap, tokenizer.decode(_fv_generate(c, kk)).strip()))
    torch.cuda.empty_cache()
    exact_accs.append(exact); ft_accs.append(ft)
    print(f"seed {s}: exact={exact:.3f}  ft={ft:.3f}  (n={n})")

print(f"\nacross {len(SEEDS)} seeds (exact-capital match, sample std ddof=1):")
print(f"  exact           : mean={np.mean(exact_accs):.3f}  std={np.std(exact_accs, ddof=1):.3f}")
print(f"  ft (cross-check): mean={np.mean(ft_accs):.3f}  std={np.std(ft_accs, ddof=1):.3f}")

# seed-0 per-country table (STRICT equality; NO alias/normalization -- judge false negatives by hand)
print("\nseed 0 per-country (strict equality; no alias table):")
print(f"{'country':24} {'gold capital':22} {'generated':22} exact")
print("-" * 78)
for country, capital, gen in seed0_rows:
    ok = gen.lower() == capital.strip().lower()
    print(f"{country[:24]:24} {capital[:22]:22} {gen[:22]:22} {ok}")

seed 0: exact=0.800  ft=0.825  (n=40)
seed 1: exact=0.675  ft=0.750  (n=40)
seed 2: exact=0.850  ft=0.925  (n=40)
seed 3: exact=0.800  ft=0.825  (n=40)
seed 4: exact=0.825  ft=0.875  (n=40)

across 5 seeds (exact-capital match, sample std ddof=1):
  exact           : mean=0.790  std=0.068
  ft (cross-check): mean=0.840  std=0.065

seed 0 per-country (strict equality; no alias table):
country                  gold capital           generated              exact
------------------------------------------------------------------------------
Laos                     Vientiane              Vientiane              True
Finland                  Helsinki               Helsinki               True
Taiwan                   Taipei                 Taipei                 True
Peru                     Lima                   Lima                   True
Fiji                     Suva                   Fiji                   False
Morocco                  Rabat                  Marrak                 False

In [10]:
# Kaggle dataset mount failed: fetch Todd's files from GitHub instead (needs Internet ON). Pinned by sha256.
import hashlib, os, urllib.request
_FV_DIR = "/kaggle/working/fv_data"
os.makedirs(_FV_DIR, exist_ok=True)
for _fn, _sha in (("antonym.json", "ba66e8e399944b557251b11a37b11f9ba7a50f0a57286f7021a8a7b26b469e8d"),
                  ("present-past.json", "bd6f7d484a967646aa5854740c70803568225f47a5630e1555c39e827ba9c78b")):
    _raw = urllib.request.urlopen("https://raw.githubusercontent.com/ericwtodd/function_vectors/main/"
                                  f"dataset_files/abstractive/{_fn}").read()
    assert hashlib.sha256(_raw).hexdigest() == _sha, f"{_fn}: upstream content changed, do not run"
    with open(f"{_FV_DIR}/{_fn}", "wb") as _f:
        _f.write(_raw)
print("data ready:", sorted(os.listdir(_FV_DIR)))


data ready: ['antonym.json', 'present-past.json']


In [11]:
# --- Cell 9 (added -- multi-task 5-seed eval; antonym + present-past + country-capital control) ---
# Applies the SAME UNIVERSAL_TOP_INDICES + EDIT_LAYER (default, unchanged) to extra tasks.
# Reuses split_data/collect/compute/clean_pred/fv_pred/first_token_id/exact_capital_match/
# _fv_generate/make_icl_prompt/framed exactly. Model load stays in Cell 2a -- never reloaded.
# Task pairs loaded FROM FILES (no inline word pairs). country_capital reuses inline PAIRS = control.
import json, os, torch, random
import numpy as np

# EDIT: point to your Kaggle Dataset mount holding Todd's dataset_files/abstractive/*.json
_DATA = "/kaggle/working/fv_data"
for _fn in ("antonym.json", "present-past.json"):        # fail fast BEFORE any model work
    assert os.path.exists(f"{_DATA}/{_fn}"), (
        f"{_DATA}/{_fn} not found -- set _DATA to your Kaggle Dataset mount "
        f"(Add Data -> function_vectors dataset; path shows under /kaggle/input/<slug>/).")
def _load_pairs(fn):
    return [(d["input"], d["output"]) for d in json.load(open(f"{_DATA}/{fn}", encoding="utf-8"))]

# PORT: country_capital reads the INLINE PAIRS (not the mounted json), deliberately, to preserve
# the validated reproduction; antonym and present_past load from the Kaggle mount via _load_pairs.
TASKS = {
    "country_capital": PAIRS,                        # inline control -> must reproduce prior numbers
    "antonym":         _load_pairs("antonym.json"),
    "present_past":    _load_pairs("present-past.json"),
}

SEEDS = [0, 1, 2, 3, 4]
for task, pairs in TASKS.items():
    if len(pairs) - N_TEST < N_ICL + 1:              # halt-and-report, no silent split shrink
        print(f"SKIP {task}: {len(pairs)} pairs, too few for N_TEST={N_TEST} + N_ICL={N_ICL}"); continue
    base_a, ft_a, ex_a = [], [], []
    seed0_rows = None
    for s in SEEDS:
        random.seed(s); np.random.seed(s); torch.manual_seed(s)
        with torch.no_grad():
            ICL_POOL, TEST = split_data(pairs, seed=s)
            mean_act  = collect_mean_head_activations(ICL_POOL, seed=s)
            fv_vector = compute_function_vector(mean_act)            # DEFAULT UNIVERSAL_TOP_INDICES
            n = len(TEST)
            base  = sum(clean_pred(x) == first_token_id(y) for x, y in TEST) / n
            ft    = sum(fv_pred(x)    == first_token_id(y) for x, y in TEST) / n
            exact = sum(exact_capital_match(x, y) for x, y in TEST) / n
            if s == 0 and task != "country_capital":             # table only for NEW tasks; skip control gens
                seed0_rows = []
                for x, y in TEST:
                    kk = len(tokenizer(" " + y, add_special_tokens=False)["input_ids"])
                    seed0_rows.append((x, y, tokenizer.decode(_fv_generate(x, kk)).strip()))
        torch.cuda.empty_cache()
        base_a.append(base); ft_a.append(ft); ex_a.append(exact)
        print(f"[{task}] seed {s}: base={base:.3f} ft={ft:.3f} exact={exact:.3f} (n={n}, pool={len(ICL_POOL)})")
    # PORT: sample std (ddof=1), not population std.
    print(f"[{task}] 5-seed (ddof=1): base={np.mean(base_a):.3f}+-{np.std(base_a, ddof=1):.3f}  "
          f"ft={np.mean(ft_a):.3f}+-{np.std(ft_a, ddof=1):.3f}  exact={np.mean(ex_a):.3f}+-{np.std(ex_a, ddof=1):.3f}")
    if task != "country_capital":                    # per-item table for NEW tasks only, seed 0
        print(f"\n[{task}] seed 0 per-item (strict equality; no alias table):")
        print(f"{'input':22} {'gold':18} {'generated':18} exact")
        print("-" * 66)
        for x, y, gen in seed0_rows:
            print(f"{x[:22]:22} {y[:18]:18} {gen[:18]:18} {gen.lower() == y.strip().lower()}")

[country_capital] seed 0: base=0.075 ft=0.825 exact=0.800 (n=40, pool=157)
[country_capital] seed 1: base=0.050 ft=0.750 exact=0.675 (n=40, pool=157)
[country_capital] seed 2: base=0.025 ft=0.925 exact=0.850 (n=40, pool=157)
[country_capital] seed 3: base=0.075 ft=0.825 exact=0.800 (n=40, pool=157)
[country_capital] seed 4: base=0.025 ft=0.875 exact=0.825 (n=40, pool=157)
[country_capital] 5-seed (ddof=1): base=0.050+-0.025  ft=0.840+-0.065  exact=0.790+-0.068
[antonym] seed 0: base=0.025 ft=0.350 exact=0.350 (n=40, pool=2358)
[antonym] seed 1: base=0.000 ft=0.300 exact=0.300 (n=40, pool=2358)
[antonym] seed 2: base=0.025 ft=0.375 exact=0.375 (n=40, pool=2358)
[antonym] seed 3: base=0.025 ft=0.400 exact=0.400 (n=40, pool=2358)
[antonym] seed 4: base=0.050 ft=0.375 exact=0.375 (n=40, pool=2358)
[antonym] 5-seed (ddof=1): base=0.025+-0.018  ft=0.360+-0.038  exact=0.360+-0.038

[antonym] seed 0 per-item (strict equality; no alias table):
input                  gold               generated

In [12]:
# --- Cell 10 (added -- read-only FV-norm diagnostic; no recompute, no model, no data load) ---
# Prints the L2 norm of each STORED function vector. Discovery: the ONLY structure that
# retains FV *tensors* is SEED_ARTIFACTS (Cell 7 / 1367d356): SEED_ARTIFACTS[seed]["fv_vector"],
# country_capital only (the 5-seed cell ran on PAIRS). Cell 9 (multi-task), Cell 6 (3-seed),
# Cell 8 (exact) stored only scores/strings -- their per-(task,seed) FV tensors were reassigned
# into the global `fv_vector` and never kept, so antonym/present_past norms cannot be shown
# without recomputation (out of scope). Tensors read exactly as stored: no device move, no
# dtype cast, no reshape. Norm is on the stored fp16 tensor (float(v.norm())); this can differ
# at the 3rd decimal from a fp32 .float().norm() (what Cell 3 printed).
import numpy as np

assert "SEED_ARTIFACTS" in globals(), "SEED_ARTIFACTS not defined -- run Cell 7 (1367d356) first"
print("Retained FV tensors: SEED_ARTIFACTS[seed]['fv_vector'] (country_capital, 5-seed cell).")
print("NOT retained (scores/strings only): antonym, present_past (Cell 9); 3-seed (Cell 6); exact (Cell 8).\n")

TASK_STORES = {"country_capital": SEED_ARTIFACTS}   # only retained FV-tensor store
for task, store in TASK_STORES.items():
    seeds = sorted(store)
    shapes = {s: tuple(store[s]["fv_vector"].shape) for s in seeds}
    norms  = {s: float(store[s]["fv_vector"].norm()) for s in seeds}   # read as stored
    uniq = set(shapes.values())
    if len(uniq) == 1:
        print(f"[{task}] shape={next(iter(uniq))} (constant across {len(seeds)} seeds)")
    else:
        print(f"[{task}] !!! SHAPE VARIES ACROSS SEEDS: {shapes}")
    print("  " + "  ".join(f"seed{s}={norms[s]:.3f}" for s in seeds))
    if set(seeds) >= {0, 1, 2, 3, 4}:
        five = [norms[s] for s in (0, 1, 2, 3, 4)]
        print(f"  all 5 seeds -> mean={np.mean(five):.3f}  std(ddof=1)={np.std(five, ddof=1):.3f}")

# built-in check: country_capital seed 0 vs standalone pre-refactor cell
if 0 in SEED_ARTIFACTS:
    v0 = SEED_ARTIFACTS[0]["fv_vector"]
    obs_norm, obs_shape = f"{float(v0.norm()):.3f}", tuple(v0.shape)
    print("\nbuilt-in check (country_capital seed 0):")
    print(f"  norm  observed={obs_norm}  expected=40.939  -> {'AGREE' if obs_norm == '40.939' else 'DIFFER'}")
    print(f"  shape observed={obs_shape}  expected=(1, 4096)  -> {'AGREE' if obs_shape == (1, 4096) else 'DIFFER'}")
else:
    print("\nbuilt-in check: seed 0 absent from SEED_ARTIFACTS -- cannot check")


Retained FV tensors: SEED_ARTIFACTS[seed]['fv_vector'] (country_capital, 5-seed cell).
NOT retained (scores/strings only): antonym, present_past (Cell 9); 3-seed (Cell 6); exact (Cell 8).

[country_capital] shape=(1, 4096) (constant across 5 seeds)
  seed0=40.938  seed1=40.688  seed2=40.969  seed3=40.500  seed4=41.188
  all 5 seeds -> mean=40.856  std(ddof=1)=0.267

built-in check (country_capital seed 0):
  norm  observed=40.938  expected=40.939  -> DIFFER
  shape observed=(1, 4096)  expected=(1, 4096)  -> AGREE


In [13]:
# --- Cell 11 (added -- FV tensor capture, all tasks x all seeds; collection only, no eval) ---
# Replicates Cell 9's (86a40179) exact per-(task,seed) extraction call path VERBATIM:
#   random.seed(s); np.random.seed(s); torch.manual_seed(s)
#   with torch.no_grad():
#       ICL_POOL, TEST = split_data(pairs, seed=s)                  # only seed=s; n_test defaulted
#       mean_act  = collect_mean_head_activations(ICL_POOL, seed=s) # only seed=s; n_trials/n_icl defaulted
#       fv_vector = compute_function_vector(mean_act)               # top_indices defaulted = UNIVERSAL_TOP_INDICES
# Stores each fv_vector in a NEW dict FV_TENSORS[(task, seed)]. Does NOT touch SEED_ARTIFACTS
# (Cell 10 reads that). No generation, no eval, no scoring, no per-item tables. Model load
# stays in Cell 2a -- never reloaded.
import torch, random
import numpy as np

assert "TASKS" in globals(), "TASKS not defined -- run Cell 9 (86a40179) first"

FV_TENSORS = {}
for task, pairs in TASKS.items():
    if len(pairs) - N_TEST < N_ICL + 1:              # same guard as Cell 9 (no task triggers it)
        print(f"SKIP {task}: {len(pairs)} pairs, too few for N_TEST={N_TEST} + N_ICL={N_ICL}"); continue
    for s in SEEDS:
        random.seed(s); np.random.seed(s); torch.manual_seed(s)
        with torch.no_grad():
            ICL_POOL, TEST = split_data(pairs, seed=s)
            mean_act  = collect_mean_head_activations(ICL_POOL, seed=s)
            fv_vector = compute_function_vector(mean_act)
        FV_TENSORS[(task, s)] = fv_vector
        torch.cuda.empty_cache()

# ---- hard stop: country_capital seed 0 must reproduce 40.939 +- 0.002 (fp32 norm) ----
_cc0 = float(FV_TENSORS[("country_capital", 0)].float().norm())
_ok  = abs(_cc0 - 40.939) <= 0.002
print(f"reproduction check: country_capital seed 0 norm observed={_cc0:.3f}  expected=40.939  -> {'AGREE' if _ok else 'DIVERGED'}")
if not _ok:
    print(f"DIVERGED by {_cc0 - 40.939:+.3f} -- loop does not reproduce Cell 9; grid suppressed, nothing else printed.")
else:
    print(f"\n{'task':16} | " + "  ".join(f"seed{s}" for s in SEEDS) + " |    mean   std(ddof=1)  shape")
    print("-" * 96)
    for task in TASKS:
        present = [s for s in SEEDS if (task, s) in FV_TENSORS]
        if not present:
            print(f"{task:16} | (skipped -- not captured)"); continue
        norms  = [float(FV_TENSORS[(task, s)].float().norm()) for s in present]
        shapes = {tuple(FV_TENSORS[(task, s)].shape) for s in present}
        shp = next(iter(shapes)) if len(shapes) == 1 else f"!!! VARIES {shapes}"
        row = "  ".join(f"{v:.3f}" for v in norms)
        print(f"{task:16} | {row} | {np.mean(norms):.3f}  {np.std(norms, ddof=1):.3f}   {shp}")

reproduction check: country_capital seed 0 norm observed=40.939  expected=40.939  -> AGREE

task             | seed0  seed1  seed2  seed3  seed4 |    mean   std(ddof=1)  shape
------------------------------------------------------------------------------------------------
country_capital  | 40.939  40.678  40.971  40.510  41.198 | 40.859  0.268   (1, 4096)
antonym          | 42.223  42.599  42.289  42.800  42.872 | 42.557  0.293   (1, 4096)
present_past     | 50.363  50.424  50.581  50.854  50.628 | 50.570  0.193   (1, 4096)


In [14]:
# --- Cell 12 (added -- 10-shot ICL accuracy per task/seed, NO function vector) ---
# Recovers the quantity Todd filters his eval on: per-item correctness under 10-shot ICL with
# NO FV and NO injection. Reproduces Cell 9's split_data(pairs, seed=s) VERBATIM so TEST is the
# identical 40 items. Demonstrations are drawn ONLY from the training pool (split_data guarantees
# pool/test disjoint), in the SAME Q:\nA: frame the notebook uses (framed()/make_icl_prompt()).
# Decoding matches _fv_generate exactly (greedy, k = gold token count, argmax->CPU, fixed-k stop)
# MINUS the injection lines. No fv_pred/_fv_generate/compute_function_vector/collect_*; no
# model.generate. Stores per-item masks in NEW dict ICL_MASK for a later filtered-FV number.
import torch, random
import numpy as np

n_icl = 10                                  # 10-shot, explicit
SEEDS_ICL = [0, 1, 2, 3, 4]
print(f"n_icl (shots) = {n_icl}")

# ---- Step 3: split-identity gate (runs BEFORE any scoring) ----
_EXP_FIRST5 = {
    "antonym":      ["simultaneous", "pedestrian", "frivolous", "swift", "dense"],
    "present_past": ["overlook", "understand", "lift", "repair", "give"],
}
_EXP_SIZES = {"country_capital": (157, 40), "antonym": (2358, 40), "present_past": (253, 40)}
gate_ok = True
print("\nsplit-identity gate (seed 0):")
for task, pairs in TASKS.items():
    pool0, test0 = split_data(pairs, seed=0)
    exp_pool, exp_test = _EXP_SIZES[task]
    size_ok = (len(pool0) == exp_pool and len(test0) == exp_test)
    gate_ok = gate_ok and size_ok
    print(f"  [{task}] pool/test = {len(pool0)}/{len(test0)}  expected {exp_pool}/{exp_test}  -> {'AGREE' if size_ok else 'DIVERGED'}")
    if task in _EXP_FIRST5:
        obs5 = [x for x, _ in test0[:5]]
        f5_ok = (obs5 == _EXP_FIRST5[task])
        gate_ok = gate_ok and f5_ok
        print(f"  [{task}] first5 observed = {obs5}")
        print(f"  [{task}] first5 expected = {_EXP_FIRST5[task]}  -> {'AGREE' if f5_ok else 'DIVERGED'}")

if not gate_ok:
    print("\nGATE DIVERGED -- split does not match Cell 9's; no scoring performed, nothing else printed.")
else:
    # ---- clean 10-shot greedy generation (NO injection); decode == _fv_generate minus injection ----
    def _icl_generate(prompt, k):
        ids = tokenizer(prompt, return_tensors="pt")["input_ids"].cuda()
        generated = []
        with torch.no_grad():
            for _ in range(k):
                with model.trace(ids):
                    logits = model.output.logits.save()          # NO fv injection
                nxt = int(logits[0, -1].argmax().cpu())          # argmax to CPU (matches _fv_generate)
                generated.append(nxt)
                ids = torch.cat([ids, torch.tensor([[nxt]], device=ids.device)], dim=1)
        return generated

    ICL_MASK = {}
    ft_by_task, ex_by_task = {t: [] for t in TASKS}, {t: [] for t in TASKS}
    for task, pairs in TASKS.items():
        for s in SEEDS_ICL:
            pool, TEST = split_data(pairs, seed=s)               # VERBATIM Cell 9 call -> identical TEST
            demo_rng = random.Random(s)                          # deterministic off the seed
            inputs, golds, ft_mask, ex_mask = [], [], [], []
            for x, y in TEST:
                demos = [f"Q: {c}\nA: {cap}" for c, cap in demo_rng.sample(pool, n_icl)]  # pool only
                prompt = "\n\n".join(demos + [framed(x)])         # same Q:\nA: frame; query tail = framed(x)
                k = len(tokenizer(" " + y, add_special_tokens=False)["input_ids"])
                gen = _icl_generate(prompt, k)
                ft = (gen[0] == first_token_id(y))
                ex = (tokenizer.decode(gen).strip().lower() == y.strip().lower())
                inputs.append(x); golds.append(y); ft_mask.append(ft); ex_mask.append(ex)
            torch.cuda.empty_cache()
            ICL_MASK[(task, s)] = {"inputs": inputs, "golds": golds, "ft": ft_mask, "exact": ex_mask}
            n = len(TEST)
            ft_acc, ex_acc = sum(ft_mask) / n, sum(ex_mask) / n
            ft_by_task[task].append(ft_acc); ex_by_task[task].append(ex_acc)
            print(f"[{task}] seed {s}: 10-shot ICL  first-token={ft_acc:.3f}  exact={ex_acc:.3f}  (n={n})")

    print("\nper-task 10-shot ICL (sample std ddof=1 when all 5 seeds present):")
    for task in TASKS:
        fts, exs = ft_by_task[task], ex_by_task[task]
        if len(fts) == len(SEEDS_ICL):
            print(f"  [{task}] first-token: mean={np.mean(fts):.3f}+-{np.std(fts, ddof=1):.3f}  "
                  f"exact: mean={np.mean(exs):.3f}+-{np.std(exs, ddof=1):.3f}")
        else:
            print(f"  [{task}] incomplete seeds ({len(fts)}/{len(SEEDS_ICL)}) -- mean/std suppressed")

n_icl (shots) = 10

split-identity gate (seed 0):
  [country_capital] pool/test = 157/40  expected 157/40  -> AGREE
  [antonym] pool/test = 2358/40  expected 2358/40  -> AGREE
  [antonym] first5 observed = ['simultaneous', 'pedestrian', 'frivolous', 'swift', 'dense']
  [antonym] first5 expected = ['simultaneous', 'pedestrian', 'frivolous', 'swift', 'dense']  -> AGREE
  [present_past] pool/test = 253/40  expected 253/40  -> AGREE
  [present_past] first5 observed = ['overlook', 'understand', 'lift', 'repair', 'give']
  [present_past] first5 expected = ['overlook', 'understand', 'lift', 'repair', 'give']  -> AGREE
[country_capital] seed 0: 10-shot ICL  first-token=0.975  exact=0.925  (n=40)
[country_capital] seed 1: 10-shot ICL  first-token=0.975  exact=0.875  (n=40)
[country_capital] seed 2: 10-shot ICL  first-token=0.975  exact=0.925  (n=40)
[country_capital] seed 3: 10-shot ICL  first-token=0.925  exact=0.900  (n=40)
[country_capital] seed 4: 10-shot ICL  first-token=0.975  exact=0.925

In [15]:
# --- Cell 13 (added -- per-item FV correctness + filtered accuracy vs ICL_MASK) ---
# Captures per-item FV correctness (FV_MASK) reproducing Cell 9's FULL FV path VERBATIM, then
# reports accuracy filtered to the items 10-shot ICL got right (Todd's filter) using ICL_MASK
# (Cell 12). first-token filtered against first-token, exact against exact -- never crossed.
# Reuses UNIVERSAL_TOP_INDICES via compute_function_vector's default (no new head selection).
# No model.generate. Writes only the NEW dict FV_MASK. Model load stays in Cell 2a.
import torch, random
import numpy as np

assert "ICL_MASK" in globals(), "ICL_MASK not defined -- run Cell 12 first"
assert "TASKS" in globals(), "TASKS not defined -- run Cell 9 first"
SEEDS_FV = [0, 1, 2, 3, 4]

# ---- reproduce Cell 9's FV path verbatim; store per-item masks ----
FV_MASK = {}
for task, pairs in TASKS.items():
    if len(pairs) - N_TEST < N_ICL + 1:              # same guard as Cell 9
        print(f"SKIP {task}: {len(pairs)} pairs, too few for N_TEST={N_TEST} + N_ICL={N_ICL}"); continue
    for s in SEEDS_FV:
        random.seed(s); np.random.seed(s); torch.manual_seed(s)
        with torch.no_grad():
            ICL_POOL, TEST = split_data(pairs, seed=s)
            mean_act  = collect_mean_head_activations(ICL_POOL, seed=s)
            fv_vector = compute_function_vector(mean_act)            # DEFAULT UNIVERSAL_TOP_INDICES
            inputs = [x for x, y in TEST]
            golds  = [y for x, y in TEST]
            ft_mask = [bool(fv_pred(x) == first_token_id(y)) for x, y in TEST]   # Cell 9 FV first-token
            ex_mask = [bool(exact_capital_match(x, y)) for x, y in TEST]         # Cell 9 FV exact
        torch.cuda.empty_cache()
        FV_MASK[(task, s)] = {"inputs": inputs, "golds": golds, "ft": ft_mask, "exact": ex_mask}

# ---- Step 3: reproduction gate (seed 0 UNFILTERED must match Cell 9) ----
_EXP = {"country_capital": (0.825, 0.800), "antonym": (0.350, 0.350), "present_past": (0.025, 0.025)}
gate_ok = True
print("reproduction gate (seed 0 unfiltered FV vs Cell 9):")
for task in TASKS:
    m = FV_MASK.get((task, 0))
    if m is None:
        print(f"  [{task}] seed 0 not captured -> DIVERGED"); gate_ok = False; continue
    n = len(m["ft"])
    ft_a, ex_a = sum(m["ft"]) / n, sum(m["exact"]) / n
    exp_ft, exp_ex = _EXP[task]
    ok = (f"{ft_a:.3f}" == f"{exp_ft:.3f}" and f"{ex_a:.3f}" == f"{exp_ex:.3f}")
    gate_ok = gate_ok and ok
    print(f"  [{task}] ft {ft_a:.3f} (exp {exp_ft:.3f})  exact {ex_a:.3f} (exp {exp_ex:.3f})  -> {'AGREE' if ok else 'DIVERGED'}")

if not gate_ok:
    print("\nREPRODUCTION GATE DIVERGED -- FV path does not reproduce Cell 9; nothing else printed.")
else:
    # ---- alignment: FV_MASK inputs must equal ICL_MASK inputs, same order ----
    align_ok = True
    for (task, s) in FV_MASK:
        if (task, s) not in ICL_MASK:
            print(f"MISALIGNED [{task} seed {s}]: no ICL_MASK entry"); align_ok = False; break
        fv_in, icl_in = FV_MASK[(task, s)]["inputs"], ICL_MASK[(task, s)]["inputs"]
        if fv_in != icl_in:
            diff = next((i for i, (a, b) in enumerate(zip(fv_in, icl_in)) if a != b), min(len(fv_in), len(icl_in)))
            fa = fv_in[diff] if diff < len(fv_in) else "<none>"
            ia = icl_in[diff] if diff < len(icl_in) else "<none>"
            print(f"MISALIGNED [{task} seed {s}] index {diff}: FV={fa!r} ICL={ia!r}"); align_ok = False; break

    if not align_ok:
        print("stopping -- FV_MASK and ICL_MASK inputs are not identical; no filtered output.")
    else:
        def _filtered(fv_bools, icl_bools):
            idx = [i for i, c in enumerate(icl_bools) if c]              # items ICL got right
            return sum(1 for i in idx if fv_bools[i]), len(idx)          # (FV-also-right, ICL-right)

        filt = {t: {"ft": [], "exact": []} for t in TASKS}
        print("\nfiltered accuracy = FV-correct among ICL-correct (numerator/denominator):")
        for task in TASKS:
            for s in SEEDS_FV:
                if (task, s) not in FV_MASK: continue
                fv, icl = FV_MASK[(task, s)], ICL_MASK[(task, s)]
                n = len(fv["ft"])
                ft_num, ft_den = _filtered(fv["ft"], icl["ft"])
                ex_num, ex_den = _filtered(fv["exact"], icl["exact"])
                filt[task]["ft"].append((ft_num, ft_den)); filt[task]["exact"].append((ex_num, ex_den))
                ft_uf, ex_uf = sum(fv["ft"]), sum(fv["exact"])
                ft_s = f"{ft_num}/{ft_den} = {ft_num/ft_den:.3f}" if ft_den else f"{ft_num}/{ft_den} = n/a (no ICL-correct)"
                ex_s = f"{ex_num}/{ex_den} = {ex_num/ex_den:.3f}" if ex_den else f"{ex_num}/{ex_den} = n/a (no ICL-correct)"
                print(f"  [{task} seed {s}] first-token filtered {ft_s}  | unfiltered {ft_uf}/{n} = {ft_uf/n:.3f}")
                print(f"  [{task} seed {s}] exact      filtered {ex_s}  | unfiltered {ex_uf}/{n} = {ex_uf/n:.3f}")

        # ---- per-task summary only when all 5 seeds present ----
        print("\nper-task filtered summary (only when all 5 seeds present):")
        for task in TASKS:
            for metric in ("ft", "exact"):
                nd = filt[task][metric]
                if len(nd) == 5:
                    fracs = [num / den for num, den in nd if den]
                    tot_num = sum(num for num, den in nd); tot_den = sum(den for num, den in nd)
                    mean_s = f"mean={np.mean(fracs):.3f}+-{np.std(fracs, ddof=1):.3f}" if len(fracs) == 5 else f"mean=n/a ({len(fracs)}/5 seeds had ICL-correct items)"
                    pooled = f"{tot_num}/{tot_den} = {tot_num/tot_den:.3f}" if tot_den else "n/a"
                    print(f"  [{task}] {metric}: per-seed {mean_s}  |  pooled {pooled}")

reproduction gate (seed 0 unfiltered FV vs Cell 9):
  [country_capital] ft 0.825 (exp 0.825)  exact 0.800 (exp 0.800)  -> AGREE
  [antonym] ft 0.350 (exp 0.350)  exact 0.350 (exp 0.350)  -> AGREE
  [present_past] ft 0.025 (exp 0.025)  exact 0.025 (exp 0.025)  -> AGREE

filtered accuracy = FV-correct among ICL-correct (numerator/denominator):
  [country_capital seed 0] first-token filtered 33/39 = 0.846  | unfiltered 33/40 = 0.825
  [country_capital seed 0] exact      filtered 31/37 = 0.838  | unfiltered 32/40 = 0.800
  [country_capital seed 1] first-token filtered 30/39 = 0.769  | unfiltered 30/40 = 0.750
  [country_capital seed 1] exact      filtered 27/35 = 0.771  | unfiltered 27/40 = 0.675
  [country_capital seed 2] first-token filtered 37/39 = 0.949  | unfiltered 37/40 = 0.925
  [country_capital seed 2] exact      filtered 33/37 = 0.892  | unfiltered 34/40 = 0.850
  [country_capital seed 3] first-token filtered 32/37 = 0.865  | unfiltered 33/40 = 0.825
  [country_capital seed 3] ex

In [16]:
# --- Cell 14 (added -- antonym pool 10-shot ICL rank-0 filter = Todd filter_set_validation) ---
# For antonym, each seed: score every POOL item with a 10-shot ICL prompt (demos sampled from the
# pool EXCLUDING that item -> no leakage) and keep items whose gold first-token is the model's
# rank-0 (argmax) prediction = Todd's filter_set_validation. Extract the FV from ONLY the kept pool
# items via collect_mean_head_activations (n_trials=100 default; everything frozen). The filter is
# clean (no FV injection). Outputs new globals; nothing existing is written; global fv_vector is
# NOT touched (local _fv). Model load stays in Cell 2a -- never reloaded.
import torch, random
import numpy as np

assert "TASKS" in globals(), "TASKS not defined -- run Cell 9 first"
FILTER_SEEDS = [0, 1, 2, 3, 4]
_task = "antonym"
_pairs = TASKS[_task]
n_icl = 10                                   # 10-shot, frozen

def _icl_rank0(query_x, gold_y, demo_pool, rng):
    # single clean forward on a 10-shot prompt; rank 0 = gold first-token is the argmax
    demos = [f"Q: {c}\nA: {cap}" for c, cap in rng.sample(demo_pool, n_icl)]
    ids = tokenizer("\n\n".join(demos + [framed(query_x)]), return_tensors="pt")["input_ids"].cuda()
    with model.trace(ids):
        logits = model.output.logits.save()
    return int(logits[0, -1].argmax()) == first_token_id(gold_y)

ANTONYM_FILTERED_FV = {}
ANTONYM_POOL_FILTER = {}
for s in FILTER_SEEDS:
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    with torch.no_grad():
        pool, TEST = split_data(_pairs, seed=s)          # same split as Cell 9; POOL is the filter target
        rng = random.Random(s)                           # deterministic demo sampling for the filter
        keep = []
        for i, (x, y) in enumerate(pool):
            others = pool[:i] + pool[i + 1:]             # demos exclude the query item (no leakage)
            keep.append(_icl_rank0(x, y, others, rng))
        filtered_pool = [pool[i] for i, k in enumerate(keep) if k]
        assert len(filtered_pool) >= n_icl + 1, f"filtered pool {len(filtered_pool)} < n_icl+1={n_icl+1}"
        mean_act = collect_mean_head_activations(filtered_pool, seed=s)   # n_trials=100 default, frozen
        _fv = compute_function_vector(mean_act)          # local -> does NOT clobber global fv_vector
    torch.cuda.empty_cache()
    ANTONYM_FILTERED_FV[s] = _fv
    ANTONYM_POOL_FILTER[s] = {"pool_size": len(pool), "kept": len(filtered_pool),
                              "keep_mask": keep, "filtered_pool": filtered_pool}
    print(f"[antonym seed {s}] pool={len(pool)} rank0-kept={len(filtered_pool)} "
          f"({len(filtered_pool) / len(pool):.3f})  |fv|={float(_fv.float().norm()):.3f}")

print(f"\nantonym filter_set_validation (10-shot ICL rank-0) done for seeds {FILTER_SEEDS}; "
      f"FVs in ANTONYM_FILTERED_FV[seed], masks in ANTONYM_POOL_FILTER[seed].")

[antonym seed 0] pool=2358 rank0-kept=1496 (0.634)  |fv|=43.892
[antonym seed 1] pool=2358 rank0-kept=1491 (0.632)  |fv|=44.098
[antonym seed 2] pool=2358 rank0-kept=1468 (0.623)  |fv|=44.112
[antonym seed 3] pool=2358 rank0-kept=1483 (0.629)  |fv|=44.289
[antonym seed 4] pool=2358 rank0-kept=1478 (0.627)  |fv|=43.692

antonym filter_set_validation (10-shot ICL rank-0) done for seeds [0, 1, 2, 3, 4]; FVs in ANTONYM_FILTERED_FV[seed], masks in ANTONYM_POOL_FILTER[seed].


In [17]:
# aqo-prerequisites-v1: read-only check. Run before aqo-extract-v2.
print("\n".join(
    f"{name} = {'present' if name in globals() else 'missing'}"
    for name in (
        "torch", "np", "model", "tokenizer",
        "TASKS", "_DATA",
        "FV_TENSORS", "ICL_MASK", "FV_MASK"
    )
))

# Modules reporting training=True (aqo-extract-v2 exempts generator / generator.streamer by exact name), and whether dropout is live.
raw = model._model  # same resolution as _aqo_runtime's raw_model
flagged = [n or "<root>" for n, m in raw.named_modules() if m.training]
print(len(flagged), flagged[:25])
print([(n, m.p) for n, m in raw.named_modules()
       if isinstance(m, torch.nn.Dropout) and m.training])
print(raw.config.embd_pdrop, raw.config.attn_pdrop, raw.config.resid_pdrop)


torch = present
np = present
model = present
tokenizer = present
TASKS = present
_DATA = present
FV_TENSORS = present
ICL_MASK = present
FV_MASK = present
2 ['generator', 'generator.streamer']
[]
0.0 0.0 0.0


In [18]:
# aqo-extract-v2: fresh query-only eligibility; original ICL_POOL remains the demo pool.
# v2 exempts nnsight's parameterless generator / generator.streamer wrappers from the eval gate by exact name.
# Producers: model/tokenizer d99d54a0; helpers/settings dd1aa448; TASKS 86a40179;
# FV_TENSORS 307ba499; ICL_MASK 06173e04; saved FV_MASK identities ba6a140f.
# Cells 7, 10, and 14 are NOT prerequisites. Legacy artifacts are never read or written.
import uuid as _aqo_uuid
ANTONYM_QUERY_ONLY_PROVENANCE = {
    "schema": "aqo-v2", "status": "in_progress", "run_id": _aqo_uuid.uuid4().hex,
}

import ast as _aqo_ast
import contextlib as _aqo_contextlib
import hashlib as _aqo_hashlib
import inspect as _aqo_inspect
import json as _aqo_json
import pathlib as _aqo_pathlib
import random as _aqo_random
import textwrap as _aqo_textwrap
import importlib.metadata as _aqo_metadata

_AQO_SEEDS = (0, 1, 2, 3, 4)
_AQO_HEADS = [(15,5),(9,14),(12,10),(8,1),(11,0),(13,13),(8,0),(14,9),(9,2),(24,6)]
# nnsight 0.7.0 LanguageModel attaches these identity wrappers after HF's .eval(), so they stay training=True.
_AQO_EVAL_EXEMPT = ("generator", "generator.streamer")
# AST hashes of the existing definitions in dd1aa448, ignoring locations/comments.
_AQO_ORIGINAL_HELPERS = {
    "as_tensor": "c202b1bfebce1dd064920c4ef12f2c74e984e40803bb91c5fe59c7a3a5591522",
    "split_data": "212c7968ecd863660b981b69cb6f947f5e5267d01e0989a89e5cfa24bbf8db29",
    "make_icl_prompt": "5ac9ca8c65c6703f7d0c68601c4a4874fc747459a748bceb7020b7bf845e52ce",
    "collect_mean_head_activations": "bc5b9700db0af3663828e9c998d8484878d174cda93c7a52945f1a56b2a29788",
    "compute_function_vector": "6ec928ac375d0daf41867385fc8c3f725e6f095847d11cc1e738ffa1228a8e9b",
    "first_token_id": "05ea364c70fc676e4ac9df09b87e53efbac0dcbf2c694cf8983f51c9e2345d05",
    "framed": "970852c743cf291f6e43aa5b14080019e0e5b2a6dd96c10a7106d070301b22f8",
    "fv_pred": "f9933c51a8bdf344cfb984ff8827e16f1592e9dc1a9ecad730f14245f8b81529",
}


def _aqo_require(ok, name, seed, condition):
    if not ok:
        raise RuntimeError(f"{name} [seed {seed}]: {condition}")


@_aqo_contextlib.contextmanager
def _aqo_prerequisite(name, seed, condition):
    try:
        yield
    except Exception as exc:
        if isinstance(exc, RuntimeError) and " [seed " in str(exc):
            raise  # preserve an already contextual validation failure
        raise RuntimeError(f"{name} [seed {seed}]: {condition} ({type(exc).__name__}: {exc})") from exc


def _aqo_digest(value):
    return _aqo_hashlib.sha256(_aqo_json.dumps(
        value, sort_keys=True, ensure_ascii=True, separators=(",", ":")
    ).encode()).hexdigest()


def _aqo_helper_stamp(fn, name):
    _aqo_require(_aqo_inspect.isfunction(fn), name, "all", "missing Python helper definition")
    try:
        tree = _aqo_ast.parse(_aqo_textwrap.dedent(_aqo_inspect.getsource(fn)))
        source_hash = _aqo_hashlib.sha256(
            _aqo_ast.dump(tree.body[0], include_attributes=False).encode()
        ).hexdigest()
    except (OSError, TypeError, SyntaxError) as exc:
        raise RuntimeError(f"{name} [seed all]: cannot fingerprint helper source") from exc
    return {"object_id": id(fn), "source_sha256": source_hash,
            "defaults": repr(fn.__defaults__), "kwdefaults": repr(fn.__kwdefaults__),
            "globals_id": id(fn.__globals__)}


def _aqo_tensor_stamp(vector, name, seed):
    _aqo_require(torch.is_tensor(vector), name, seed, "missing tensor")
    _aqo_require(tuple(vector.shape) == (1, 4096), name, seed, "expected shape (1, 4096)")
    _aqo_require(vector.dtype == torch.float16, name, seed, "expected original float16 dtype")
    _aqo_require(bool(torch.isfinite(vector).all()), name, seed, "non-finite tensor")
    raw = vector.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes()
    return {"sha256": _aqo_hashlib.sha256(raw).hexdigest(), "shape": list(vector.shape),
            "dtype": str(vector.dtype), "device": str(vector.device)}


def _aqo_saved_eval(seed):
    key = ("antonym", seed)
    for name in ("FV_MASK", "ICL_MASK"):
        store = globals().get(name)
        _aqo_require(isinstance(store, dict) and isinstance(store.get(key), dict),
                     name, seed, "missing antonym entry from its producer cell")
        for field in ("inputs", "golds"):
            values = store[key].get(field)
            _aqo_require(isinstance(values, (list, tuple)) and len(values) == 40
                         and all(isinstance(x, str) for x in values),
                         f"{name}.{field}", seed, "expected 40 ordered strings")
    fv, icl = FV_MASK[key], ICL_MASK[key]
    for field in ("inputs", "golds"):
        _aqo_require(list(fv[field]) == list(icl[field]), f"FV_MASK/ICL_MASK.{field}",
                     seed, "saved ordered identities differ")
    mask = icl.get("ft")
    _aqo_require(isinstance(mask, (list, tuple, np.ndarray))
                 and (not isinstance(mask, np.ndarray) or mask.ndim == 1) and len(mask) == 40
                 and all(isinstance(x, (bool, np.bool_)) for x in mask),
                 "ICL_MASK.ft", seed, "expected full-length Boolean mask, not filtered rows")
    _aqo_require(any(mask), "ICL_MASK.ft", seed, "empty retained evaluation set")
    return {"inputs": list(fv["inputs"]), "golds": list(fv["golds"]),
            "ft": [bool(x) for x in mask]}


def _aqo_split_rows(pairs, seed):
    # ID is the row's position in the original dataset, so duplicate-valued rows stay distinct.
    order = list(range(len(pairs)))
    _aqo_random.Random(seed).shuffle(order)
    rows = [(i, pairs[i][0], pairs[i][1]) for i in order]
    return rows[40:], rows[:40]


def _aqo_runtime():
    for name in ("torch", "np", "model", "tokenizer", "TASKS", "_DATA"):
        _aqo_require(name in globals(), name, "all", "missing prerequisite")
    settings = {"N_HEADS": 16, "HEAD_DIM": 256, "RESID_DIM": 4096, "N_LAYERS": 28,
                "EDIT_LAYER": 9, "N_ICL": 10, "N_TRIALS": 100, "N_TEST": 40, "SEED": 0}
    for name, value in settings.items():
        _aqo_require(globals().get(name) == value, name, "all", f"locked value must be {value}")
    _aqo_require(globals().get("UNIVERSAL_TOP_INDICES") == _AQO_HEADS,
                 "UNIVERSAL_TOP_INDICES", "all", "published head order changed")
    settings["UNIVERSAL_TOP_INDICES"] = [list(x) for x in _AQO_HEADS]
    versions = {}
    for name in ("nnsight", "torch", "transformers"):
        with _aqo_prerequisite(name, "all", "installed version metadata unavailable"):
            versions[name] = _aqo_metadata.version(name)
    _aqo_require(versions["nnsight"] == "0.7.0", "nnsight", "all", "requires version 0.7.0")
    helpers = {}
    for name, expected in _AQO_ORIGINAL_HELPERS.items():
        fn = globals().get(name)
        stamp = _aqo_helper_stamp(fn, name)
        _aqo_require(stamp["source_sha256"] == expected, name, "all", "definition differs from dd1aa448")
        # Validate the actual namespace used by each helper, including fv_pred's globals.
        for dep in fn.__code__.co_names:
            if dep in (*_AQO_ORIGINAL_HELPERS, *settings, "model", "tokenizer", "torch"):
                _aqo_require(fn.__globals__.get(dep) is globals().get(dep), name, "all",
                             f"global binding {dep} differs from the notebook")
        helpers[name] = stamp
    _aqo_require(split_data.__defaults__ == (40, 0), "split_data", "all", "defaults changed")
    _aqo_require(collect_mean_head_activations.__defaults__ == (100, 10, 0),
                 "collect_mean_head_activations", "all", "defaults changed")
    _aqo_require(compute_function_vector.__defaults__ == (_AQO_HEADS,),
                 "compute_function_vector", "all", "published default heads changed")
    for name, fn in list(globals().items()):
        if name.startswith("_aqo_") and _aqo_inspect.isfunction(fn) and name not in _AQO_EVAL_HELPERS:
            helpers[name] = _aqo_helper_stamp(fn, name)
    with _aqo_prerequisite("model", "all", "loaded model/config/parameters unavailable or inconsistent"):
        raw_model = model._model
        cfg = raw_model.config
        _aqo_require((cfg.model_type, cfg.n_layer, cfg.n_head, cfg.n_embd) == ("gptj", 28, 16, 4096),
                     "model.config", "all", "expected GPT-J 28 layers, 16 heads, hidden 4096")
        # Every alias is checked under each name; exemption is an exact-name match, never a prefix.
        modules = list(raw_model.named_modules(remove_duplicate=False))
        training = [n or "<root>" for n, m in modules if m.training and n not in _AQO_EVAL_EXEMPT]
        _aqo_require(not training, "model", "all", f"must be in eval mode; training=True: {training}")
        exempt = {n: m for n, m in modules if n in _AQO_EVAL_EXEMPT}
        for n, m in exempt.items():  # parameters()/buffers() recurse, so no exemption can hide weights
            _aqo_require(next(m.parameters(), None) is None and next(m.buffers(), None) is None,
                         n, "all", "eval-exempt subtree must hold zero parameters and zero buffers")
        parameter = next(raw_model.parameters(), None)
        _aqo_require(parameter is not None, "model", "all", "loaded model has no parameters")
        _aqo_require(parameter.device.type == "cuda" and parameter.dtype == torch.float16,
                     "model", "all", "requires the existing CUDA float16 model")
        # Version counters catch normal in-place weight changes without copying the 12 GB model.
        weights = [(name, id(p), p._version, str(p.dtype), str(p.device), list(p.shape))
                   for name, p in raw_model.named_parameters()]
        model_identity = {"object_id": id(model), "raw_object_id": id(raw_model),
                          "name": cfg._name_or_path, "config_sha256": _aqo_digest(cfg.to_dict()),
                          "weights_version_sha256": _aqo_digest(weights), "device": str(parameter.device),
                          "eval_exempt": {n: bool(m.training) for n, m in exempt.items()}}
    with _aqo_prerequisite("tokenizer", "all", "loaded tokenizer identity/configuration unavailable or inconsistent"):
        _aqo_require(tokenizer is model.tokenizer, "tokenizer", "all", "must be the loaded model tokenizer")
        tokens = {"vocab": tokenizer.get_vocab(), "special_tokens": tokenizer.special_tokens_map,
                  "padding_side": tokenizer.padding_side, "truncation_side": tokenizer.truncation_side,
                  "model_max_length": tokenizer.model_max_length}
        tokenizer_identity = {"object_id": id(tokenizer), "name": tokenizer.name_or_path,
                              "sha256": _aqo_digest(tokens)}
    return {"settings": settings, "versions": versions, "helpers": helpers,
            "model": model_identity, "tokenizer": tokenizer_identity}


def _aqo_snapshot():
    runtime = _aqo_runtime()
    pairs = TASKS.get("antonym") if isinstance(TASKS, dict) else None
    _aqo_require(isinstance(pairs, (list, tuple)) and len(pairs) >= 51, "TASKS.antonym", "all",
                 "requires 40 test rows plus at least 11 pool rows")
    _aqo_require(all(isinstance(p, (list, tuple)) and len(p) == 2
                     and all(isinstance(v, str) for v in p) for p in pairs),
                 "TASKS.antonym", "all", "expected input/gold string pairs")
    with _aqo_prerequisite("_DATA/antonym.json", "all", "source file unavailable or invalid input/output JSON"):
        path = _aqo_pathlib.Path(_DATA) / "antonym.json"
        _aqo_require(path.is_file(), "_DATA/antonym.json", "all", f"source file unavailable: {path}")
        raw = path.read_bytes()
        source_pairs = [(row["input"], row["output"]) for row in _aqo_json.loads(raw)]
    _aqo_require([tuple(p) for p in pairs] == source_pairs, "TASKS.antonym", "all",
                 "ordered identities differ from source antonym.json")
    snapshot = {"runtime": runtime,
                "dataset": {"path": str(path.resolve()), "file_sha256": _aqo_hashlib.sha256(raw).hexdigest(),
                            "rows": [[i, x, y] for i, (x, y) in enumerate(pairs)]}, "seeds": {}}
    for seed in _AQO_SEEDS:
        evaluation = _aqo_saved_eval(seed)
        pool, test = _aqo_split_rows(pairs, seed)
        actual_pool, actual_test = split_data(pairs, seed=seed)
        _aqo_require([tuple(p) for p in actual_pool] == [(x, y) for _, x, y in pool]
                     and [tuple(p) for p in actual_test] == [(x, y) for _, x, y in test],
                     "split_data", seed, "does not match original local random.Random(seed) shuffle")
        _aqo_require(evaluation["inputs"] == [x for _, x, _ in test]
                     and evaluation["golds"] == [y for _, _, y in test],
                     "FV_MASK", seed, "saved TEST identities differ from reconstructed split")
        store = globals().get("FV_TENSORS")
        _aqo_require(isinstance(store, dict) and ("antonym", seed) in store,
                     "FV_TENSORS", seed, "missing original antonym vector")
        fingerprint = _aqo_tensor_stamp(store[("antonym", seed)], "FV_TENSORS", seed)
        _aqo_require(fingerprint["device"] == runtime["model"]["device"], "FV_TENSORS", seed,
                     "vector device differs from model")
        snapshot["seeds"][seed] = {"pool": [list(r) for r in pool], "test": [list(r) for r in test],
                                    "evaluation": evaluation, "original_tensor": fingerprint}
    return snapshot


def _aqo_check_snapshot(expected):
    actual = _aqo_snapshot()
    for name in ("runtime", "dataset"):
        _aqo_require(actual[name] == expected.get(name), name, "all", "changed since extraction preflight")
    for seed in _AQO_SEEDS:
        _aqo_require(actual["seeds"][seed] == expected.get("seeds", {}).get(seed),
                     "evaluation/split/FV_TENSORS", seed, "changed since extraction preflight")


def _aqo_prompt(demos, query):
    return "\n\n".join([f"Q: {x}\nA: {y}" for _, x, y in demos] + [framed(query[1])])


def _aqo_filter_queries(demo_pool, query_pool, seed):
    rng = _aqo_random.Random(seed)  # separate from split RNG and extraction RNG
    decisions = []
    for query in query_pool:
        others = [row for row in demo_pool if row[0] != query[0]]
        demos = rng.sample(others, 10)  # full pool, without replacement, exclude only this row ID
        ids = tokenizer(_aqo_prompt(demos, query), return_tensors="pt")["input_ids"].cuda()
        with torch.no_grad(), model.trace(ids):
            logits = model.output.logits.save()  # clean 10-shot, no FV injection
        predicted, gold = int(logits[0, -1].argmax().cpu()), first_token_id(query[2])
        decisions.append({"query_id": query[0], "demo_ids": [row[0] for row in demos],
                          "predicted_id": predicted, "gold_id": gold, "eligible": predicted == gold})
    return decisions


def _aqo_trials(demo_pool, query_pool, seed, n_trials=100):
    _aqo_require(len(demo_pool) >= 11, "demo_pool", seed, "requires at least 11 full-pool rows")
    by_id = {row[0]: tuple(row) for row in demo_pool}
    _aqo_require(len(by_id) == len(demo_pool), "demo_pool", seed, "duplicate row IDs")
    _aqo_require(len(query_pool) > 0, "query_pool", seed, "no eligible extraction queries")
    _aqo_require(all(by_id.get(row[0]) == tuple(row) for row in query_pool),
                 "query_pool", seed, "eligible row IDs/identities must belong to full demo pool")
    eligible_ids = {row[0] for row in query_pool}
    rng = _aqo_random.Random(seed)
    proposals, accepted = 0, 0
    while accepted < n_trials:
        proposal = rng.sample(demo_pool, 11)
        proposals += 1
        if proposal[-1][0] not in eligible_ids:
            continue  # reject BEFORE prompt construction, tokenization, or inference
        accepted += 1
        yield proposal[:10], proposal[-1], proposals


def _aqo_collect(demo_pool, query_pool, seed, ledger):
    acc = torch.zeros(N_LAYERS, N_HEADS, HEAD_DIM, dtype=torch.float32, device="cpu")
    for demos, query, proposals in _aqo_trials(demo_pool, query_pool, seed, n_trials=100):
        ids = tokenizer(_aqo_prompt(demos, query), return_tensors="pt")["input_ids"].cuda()
        saved = []
        with torch.no_grad(), model.trace(ids):
            for layer in range(N_LAYERS):  # ascending activation-hook order
                saved.append(model.transformer.h[layer].attn.out_proj.input[:, -1, :].save())
        for layer in range(N_LAYERS):
            last = as_tensor(saved[layer]).cpu().float()
            acc[layer] += last.view(N_HEADS, HEAD_DIM)
        ledger.append({"query_id": query[0], "demo_ids": [row[0] for row in demos],
                       "proposal_number": proposals})
    return (acc / 100).to("cuda")  # existing compute_function_vector contract


def _aqo_no_alias(old_vectors, new_vectors):
    for seed, new in new_vectors.items():
        for old_seed, old in old_vectors.items():
            same_storage = (new.device == old.device and
                            new.untyped_storage().data_ptr() == old.untyped_storage().data_ptr())
            _aqo_require(new is not old and not same_storage, "ANTONYM_QUERY_ONLY_FV", seed,
                         f"aliases original FV_TENSORS seed {old_seed}")


def _aqo_seal(provenance):
    return _aqo_digest({k: v for k, v in provenance.items() if k not in ("status", "seal_sha256")})


def _aqo_extract_run(provenance):
    # ALL five seeds pass validation before the first tokenization or model inference.
    snapshot = _aqo_snapshot()
    provenance.update({"snapshot": snapshot, "seeds": {},
        "producers": {"model_tokenizer": "d99d54a0", "helpers_settings": "dd1aa448",
                      "task_data": "86a40179", "original_vectors": "307ba499",
                      "evaluation_eligibility": "06173e04", "saved_identities": "ba6a140f"},
        "sampling": "Original joint sample(full_pool, 11), conditioned on eligible last/query row.",
        "pairing": "Seed labels are paired; accepted trials are not individually matched to original trials. "
                   "Matching seed labels does not establish matching demonstration sequences.",
        "identity_limits": "Original tensors and saved evaluation artifacts have no historical provenance; "
                           "their identities/content are locked at this run's preflight. Model weights are "
                           "tracked by parameter identity/version counters, not a full weight-content hash."})
    vectors = {}
    for seed in _AQO_SEEDS:
        provenance["active_seed"] = seed
        pool = snapshot["seeds"][seed]["pool"]  # original ICL_POOL for this task/seed
        _aqo_random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        decisions = _aqo_filter_queries(demo_pool=pool, query_pool=pool, seed=seed)
        query_pool = [row for row, decision in zip(pool, decisions) if decision["eligible"]]
        record = {"run_id": provenance["run_id"], "filter_decisions": decisions,
                  "eligible_query_ids": [row[0] for row in query_pool], "trials": [],
                  "seed_handling": {"split_local_random": seed, "filter_local_random": seed,
                                    "extraction_local_random": seed,
                                    "global_python_numpy_torch": seed,
                                    "global_resets_before": ["filter", "extraction"]}}
        provenance["seeds"][seed] = record
        _aqo_require(bool(query_pool), "query_pool", seed, "no eligible extraction queries")
        _aqo_random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        with torch.no_grad():
            mean_act = _aqo_collect(demo_pool=pool, query_pool=query_pool, seed=seed, ledger=record["trials"])
            vector = compute_function_vector(mean_act)  # original function and published default heads
        record["tensor"] = _aqo_tensor_stamp(vector, "ANTONYM_QUERY_ONLY_FV", seed)
        record["accepted_trials"] = len(record["trials"])
        record["proposal_count"] = record["trials"][-1]["proposal_number"]
        _aqo_require(record["accepted_trials"] == 100, "trials", seed, "requires 100 accepted trials")
        vectors[seed] = vector
        print(f"[antonym seed {seed}] eligible queries={len(query_pool)}/{len(pool)}; "
              f"demos from all {len(pool)} rows; accepted=100 proposals={record['proposal_count']}")
    _aqo_check_snapshot(snapshot)
    _aqo_no_alias({s: FV_TENSORS[("antonym", s)] for s in _AQO_SEEDS}, vectors)
    provenance.pop("active_seed", None)
    provenance["seal_sha256"] = _aqo_seal(provenance)
    return vectors


# Evaluation helpers may already exist if these two cells are rerun in the same kernel.
_AQO_EVAL_HELPERS = {"_aqo_validate_run", "_aqo_score", "_aqo_paired_stats", "_aqo_evaluate_run"}
try:
    _aqo_completed_vectors = _aqo_extract_run(ANTONYM_QUERY_ONLY_PROVENANCE)
    ANTONYM_QUERY_ONLY_FV = _aqo_completed_vectors  # publish only after complete five-seed success
    ANTONYM_QUERY_ONLY_PROVENANCE["status"] = "complete"
    del _aqo_completed_vectors
    print("Published ANTONYM_QUERY_ONLY_FV for all five seeds. "
          "Accepted trials are not individually matched to the original trials.")
except BaseException as _aqo_error:
    ANTONYM_QUERY_ONLY_PROVENANCE.update(status="failed", error=f"{type(_aqo_error).__name__}: {_aqo_error}")
    raise


[antonym seed 0] eligible queries=1496/2358; demos from all 2358 rows; accepted=100 proposals=153
[antonym seed 1] eligible queries=1491/2358; demos from all 2358 rows; accepted=100 proposals=155
[antonym seed 2] eligible queries=1468/2358; demos from all 2358 rows; accepted=100 proposals=150
[antonym seed 3] eligible queries=1483/2358; demos from all 2358 rows; accepted=100 proposals=158
[antonym seed 4] eligible queries=1478/2358; demos from all 2358 rows; accepted=100 proposals=160
Published ANTONYM_QUERY_ONLY_FV for all five seeds. Accepted trials are not individually matched to the original trials.


In [19]:
# aqo-evaluate-v1: paired FIRST-TOKEN accuracy on saved 10-shot-eligible evaluation rows.
# Cell 13's FV_MASK correctness arrays are unfiltered and lack vector provenance.
# Use only its ordered identities; rescore BOTH original and query-only vectors on all 40.
import uuid as _aqo_eval_uuid
ANTONYM_QUERY_ONLY_RESULTS = {"status": "in_progress", "evaluation_id": _aqo_eval_uuid.uuid4().hex}


def _aqo_validate_run():
    provenance = globals().get("ANTONYM_QUERY_ONLY_PROVENANCE")
    _aqo_require(isinstance(provenance, dict) and provenance.get("status") == "complete",
                 "ANTONYM_QUERY_ONLY_PROVENANCE", "all", "requires a complete extraction run")
    _aqo_require(provenance.get("schema") == "aqo-v2" and bool(provenance.get("run_id")),
                 "ANTONYM_QUERY_ONLY_PROVENANCE", "all", "missing schema/run ID")
    _aqo_require(provenance.get("seal_sha256") == _aqo_seal(provenance),
                 "ANTONYM_QUERY_ONLY_PROVENANCE", "all", "provenance content changed or incomplete")
    _aqo_require(set(provenance.get("seeds", {})) == set(_AQO_SEEDS),
                 "ANTONYM_QUERY_ONLY_PROVENANCE", "all", "requires all five seeds")
    vectors = globals().get("ANTONYM_QUERY_ONLY_FV")
    _aqo_require(isinstance(vectors, dict) and set(vectors) == set(_AQO_SEEDS),
                 "ANTONYM_QUERY_ONLY_FV", "all", "requires exactly the five extraction seeds")
    _aqo_check_snapshot(provenance["snapshot"])
    for seed in _AQO_SEEDS:
        record = provenance["seeds"][seed]
        _aqo_require(record.get("run_id") == provenance["run_id"] and record.get("accepted_trials") == 100
                     and len(record.get("trials", [])) == 100,
                     "ANTONYM_QUERY_ONLY_PROVENANCE", seed, "incomplete or stale seed run")
        _aqo_require(_aqo_tensor_stamp(vectors[seed], "ANTONYM_QUERY_ONLY_FV", seed) == record.get("tensor"),
                     "ANTONYM_QUERY_ONLY_FV", seed, "tensor changed since extraction")
    _aqo_no_alias({s: FV_TENSORS[("antonym", s)] for s in _AQO_SEEDS}, vectors)
    return provenance


def _aqo_score(vector, inputs, golds):
    scorer = fv_pred
    namespace = scorer.__globals__  # bind the namespace ACTUALLY read by the existing scorer
    absent = object()
    previous = namespace.get("fv_vector", absent)
    try:
        namespace["fv_vector"] = vector
        with torch.no_grad():
            # fv_pred applies framed(x), a zero-shot Q:/A: prompt, internally.
            return [bool(scorer(x) == first_token_id(y)) for x, y in zip(inputs, golds)]
    finally:
        if previous is absent:
            namespace.pop("fv_vector", None)
        else:
            namespace["fv_vector"] = previous


def _aqo_paired_stats(rows):
    old = np.array([rows[s]["old_accuracy"] for s in _AQO_SEEDS], dtype=float)
    new = np.array([rows[s]["new_accuracy"] for s in _AQO_SEEDS], dtype=float)
    differences = new - old
    return {"old_mean": float(old.mean()), "old_std": float(old.std(ddof=1)),
            "new_mean": float(new.mean()), "new_std": float(new.std(ddof=1)),
            "paired_differences": differences.tolist(), "difference_mean": float(differences.mean()),
            "difference_std": float(differences.std(ddof=1)), "ddof": 1, "weighting": "unweighted seeds",
            "reported_old_filtered_mean": 0.548, "old_agrees_at_reported_precision": f"{old.mean():.3f}" == "0.548"}


def _aqo_evaluate_run():
    provenance = _aqo_validate_run()  # all seeds, tensors, saved identities/masks, helpers/settings first
    rows = {}
    for seed in _AQO_SEEDS:
        evaluation = provenance["snapshot"]["seeds"][seed]["evaluation"]
        inputs, golds, mask = evaluation["inputs"], evaluation["golds"], evaluation["ft"]
        retained = [i for i, keep in enumerate(mask) if keep]  # derive ONCE from saved full mask
        old_full = _aqo_score(FV_TENSORS[("antonym", seed)], inputs, golds)
        new_full = _aqo_score(ANTONYM_QUERY_ONLY_FV[seed], inputs, golds)
        old_kept, new_kept = [old_full[i] for i in retained], [new_full[i] for i in retained]
        old_count, new_count, denominator = sum(old_kept), sum(new_kept), len(retained)
        rows[seed] = {"run_id": provenance["run_id"], "inputs": list(inputs), "golds": list(golds),
                      "ft_mask": list(mask), "retained_indexes": retained, "full_count": len(inputs),
                      "old_full_correct": old_full, "new_full_correct": new_full,
                      "retained_count": denominator, "old_correct": old_count, "new_correct": new_count,
                      "old_accuracy": old_count / denominator, "new_accuracy": new_count / denominator,
                      "difference": (new_count - old_count) / denominator,
                      "original_tensor": provenance["snapshot"]["seeds"][seed]["original_tensor"],
                      "query_only_tensor": provenance["seeds"][seed]["tensor"]}
    _aqo_validate_run()  # detect any accidental mutation during scoring before publishing results
    return {"run_id": provenance["run_id"], "seeds": rows, "summary": _aqo_paired_stats(rows),
            "extraction_seal_sha256": provenance["seal_sha256"]}


try:
    if "_aqo_require" not in globals():
        raise RuntimeError("aqo-extract-v2 [seed all]: helper definitions unavailable; run extraction first")
    _aqo_evaluated = _aqo_evaluate_run()
    ANTONYM_QUERY_ONLY_RESULTS.update(_aqo_evaluated, status="complete")
    del _aqo_evaluated
    for _aqo_s, _aqo_row in ANTONYM_QUERY_ONLY_RESULTS["seeds"].items():
        print(f"[antonym seed {_aqo_s}] retained={_aqo_row['retained_count']}/40 "
              f"old={_aqo_row['old_correct']}/{_aqo_row['retained_count']} ({_aqo_row['old_accuracy']:.3f}) "
              f"new={_aqo_row['new_correct']}/{_aqo_row['retained_count']} ({_aqo_row['new_accuracy']:.3f}) "
              f"new - old={_aqo_row['difference']:+.3f}")
    _aqo_summary = ANTONYM_QUERY_ONLY_RESULTS["summary"]
    print("\nUnweighted five-seed means and sample standard deviations (ddof=1):")
    for _aqo_label in ("old", "new", "difference"):
        print(f"  {_aqo_label}: mean={_aqo_summary[_aqo_label + '_mean']:.6f} "
              f"std={_aqo_summary[_aqo_label + '_std']:.6f}")
    print(f"  paired differences (new - old): {_aqo_summary['paired_differences']}")
    _aqo_agrees = _aqo_summary["old_agrees_at_reported_precision"]
    print(f"Computed old filtered mean={_aqo_summary['old_mean']:.6f}; reported=0.548: "
          + ("AGREE at reported precision." if _aqo_agrees else "DISAGREEMENT; reproduction is not established."))
except BaseException as _aqo_eval_error:
    ANTONYM_QUERY_ONLY_RESULTS.update(status="failed", error=f"{type(_aqo_eval_error).__name__}: {_aqo_eval_error}")
    raise


[antonym seed 0] retained=25/40 old=13/25 (0.520) new=13/25 (0.520) new - old=+0.000
[antonym seed 1] retained=20/40 old=11/20 (0.550) new=12/20 (0.600) new - old=+0.050
[antonym seed 2] retained=22/40 old=13/22 (0.591) new=13/22 (0.591) new - old=+0.000
[antonym seed 3] retained=26/40 old=14/26 (0.538) new=14/26 (0.538) new - old=+0.000
[antonym seed 4] retained=26/40 old=14/26 (0.538) new=14/26 (0.538) new - old=+0.000

Unweighted five-seed means and sample standard deviations (ddof=1):
  old: mean=0.547566 std=0.026506
  new: mean=0.557566 std=0.035544
  difference: mean=0.010000 std=0.022361
  paired differences (new - old): [0.0, 0.04999999999999993, 0.0, 0.0, 0.0]
Computed old filtered mean=0.547566; reported=0.548: AGREE at reported precision.


In [20]:
# aqo-cosine-v1: read-only per-seed cosine between the original and query-only antonym vectors.
# FV_TENSORS is keyed by ("antonym", seed); ANTONYM_QUERY_ONLY_FV by the bare int seed. Defines no _aqo_ functions.
if "_aqo_validate_run" not in globals():
    raise RuntimeError("aqo-evaluate-v1 [seed all]: validation helpers unavailable; "
                       "run aqo-extract-v2 and aqo-evaluate-v1 first")
_aqo_validate_run()  # sealed provenance, all five seeds, unchanged tensors, no aliasing; raises otherwise
ANTONYM_QUERY_ONLY_COSINE = {_aqo_s: float(torch.nn.functional.cosine_similarity(
    FV_TENSORS[("antonym", _aqo_s)].float(), ANTONYM_QUERY_ONLY_FV[_aqo_s].float(), dim=-1))
    for _aqo_s in _AQO_SEEDS}  # published only after every seed is computed
for _aqo_s, _aqo_cos in ANTONYM_QUERY_ONLY_COSINE.items():
    print(f"[antonym seed {_aqo_s}] cos(original, query-only) = {_aqo_cos:.6f}")
_aqo_cos = np.array(list(ANTONYM_QUERY_ONLY_COSINE.values()), dtype=float)
print(f"Unweighted five-seed cosine: mean={_aqo_cos.mean():.6f} std={_aqo_cos.std(ddof=1):.6f} (ddof=1)")


[antonym seed 0] cos(original, query-only) = 0.998741
[antonym seed 1] cos(original, query-only) = 0.998196
[antonym seed 2] cos(original, query-only) = 0.998041
[antonym seed 3] cos(original, query-only) = 0.998539
[antonym seed 4] cos(original, query-only) = 0.998505
Unweighted five-seed cosine: mean=0.998404 std=0.000282 (ddof=1)


In [21]:
# aqo-diagnostics-v1: read-only antonym cross-seed and norm-ratio baselines, present_past seed-0 cosines, and old/new
# concordance on the retained rows aqo-evaluate-v1 scored. No inference, .float() copies only, no _aqo_ functions.
if "_aqo_validate_run" not in globals():
    raise RuntimeError("aqo-evaluate-v1 [seed all]: validation helpers unavailable; "
                       "run aqo-extract-v2 and aqo-evaluate-v1 first")
_aqo_diag_prov = _aqo_validate_run()  # sealed provenance, both antonym arms unchanged, no aliasing; raises otherwise
_aqo_diag_pp = {_aqo_s: _aqo_tensor_stamp(FV_TENSORS.get(("present_past", _aqo_s)), "FV_TENSORS.present_past", _aqo_s)
                for _aqo_s in _AQO_SEEDS}  # (1, 4096) float16 and finite, else a contextual RuntimeError
_aqo_diag_pairs = [(_aqo_a, _aqo_b) for _aqo_a in _AQO_SEEDS for _aqo_b in _AQO_SEEDS if _aqo_a < _aqo_b]
_aqo_diag_cos = {(_aqo_t, _aqo_p): float(torch.nn.functional.cosine_similarity(
    FV_TENSORS[(_aqo_t, _aqo_p[0])].float(), FV_TENSORS[(_aqo_t, _aqo_p[1])].float(), dim=-1))
    for _aqo_t in ("antonym", "present_past") for _aqo_p in _aqo_diag_pairs}
_aqo_diag_norms = {_aqo_s: (float(FV_TENSORS[("antonym", _aqo_s)].float().norm()),
                            float(ANTONYM_QUERY_ONLY_FV[_aqo_s].float().norm())) for _aqo_s in _AQO_SEEDS}
_aqo_require(all(_aqo_n[0] > 0 for _aqo_n in _aqo_diag_norms.values()), "FV_TENSORS", "all", "zero-norm antonym vector")
# 1. Antonym cross-seed baseline: original vs original, all 10 pairs a < b.
_aqo_diag_cross = {_aqo_p: _aqo_c for (_aqo_t, _aqo_p), _aqo_c in _aqo_diag_cos.items() if _aqo_t == "antonym"}
for (_aqo_diag_a, _aqo_diag_b), _aqo_diag_c in _aqo_diag_cross.items():
    print(f"[antonym seeds {_aqo_diag_a}-{_aqo_diag_b}] cos(original, original) = {_aqo_diag_c:.6f}")
_aqo_diag_summary = {"min": min(_aqo_diag_cross.values()), "mean": sum(_aqo_diag_cross.values()) / len(_aqo_diag_cross),
                     "max": max(_aqo_diag_cross.values())}
print(f"antonym cross-seed cos over {len(_aqo_diag_cross)} pairs: min={_aqo_diag_summary['min']:.6f} "
      f"mean={_aqo_diag_summary['mean']:.6f} max={_aqo_diag_summary['max']:.6f}")
# 2. Antonym norm ratio per seed.
_aqo_diag_ratio = {_aqo_s: {"original": _aqo_o, "query_only": _aqo_q, "ratio": _aqo_q / _aqo_o}
                   for _aqo_s, (_aqo_o, _aqo_q) in _aqo_diag_norms.items()}
for _aqo_diag_s, _aqo_diag_r in _aqo_diag_ratio.items():
    print(f"[antonym seed {_aqo_diag_s}] norm(original) = {_aqo_diag_r['original']:.6f} "
          f"norm(query-only) = {_aqo_diag_r['query_only']:.6f} query-only/original = {_aqo_diag_r['ratio']:.6f}")
# 3. present_past: seed 0 against seeds 1-4, then the 6 pairs among seeds 1-4, printed as separate groups.
_aqo_diag_pp_cos = {_aqo_p: _aqo_c for (_aqo_t, _aqo_p), _aqo_c in _aqo_diag_cos.items() if _aqo_t == "present_past"}
_aqo_diag_groups = {
    "seed 0 vs seeds 1-4": {_aqo_p: _aqo_c for _aqo_p, _aqo_c in _aqo_diag_pp_cos.items() if _aqo_p[0] == 0},
    "pairs among seeds 1-4": {_aqo_p: _aqo_c for _aqo_p, _aqo_c in _aqo_diag_pp_cos.items() if _aqo_p[0] > 0}}
for _aqo_diag_g, _aqo_diag_group in _aqo_diag_groups.items():
    print(f"present_past {_aqo_diag_g}:")
    for (_aqo_diag_a, _aqo_diag_b), _aqo_diag_c in _aqo_diag_group.items():
        print(f"  [present_past seeds {_aqo_diag_a}-{_aqo_diag_b}] cos(original, original) = {_aqo_diag_c:.6f}")
# 4. Concordance, only from per-item correctness the paired evaluation already kept for THIS sealed run.
_AQO_DIAG_EXPECTED = {0: (25, 13, 13), 1: (20, 11, 12), 2: (22, 13, 13), 3: (26, 14, 14), 4: (26, 14, 14)}
_aqo_diag_res = globals().get("ANTONYM_QUERY_ONLY_RESULTS")
_aqo_diag_status = "computed" if (
    isinstance(_aqo_diag_res, dict) and _aqo_diag_res.get("status") == "complete"
    and _aqo_diag_res.get("run_id") == _aqo_diag_prov["run_id"]
    and _aqo_diag_res.get("extraction_seal_sha256") == _aqo_diag_prov["seal_sha256"]
    and isinstance(_aqo_diag_res.get("seeds"), dict) and set(_aqo_diag_res["seeds"]) == set(_AQO_SEEDS)
    and all(isinstance(_aqo_r, dict) for _aqo_r in _aqo_diag_res["seeds"].values())
) else "paired evaluation results unavailable"
for _aqo_diag_s in (_AQO_SEEDS if _aqo_diag_status == "computed" else ()):
    _aqo_diag_row = _aqo_diag_res["seeds"][_aqo_diag_s]
    _aqo_diag_kept = [isinstance(_aqo_diag_row.get(_aqo_k), list) and len(_aqo_diag_row[_aqo_k]) == 40
                      and all(isinstance(_aqo_x, (bool, np.bool_)) for _aqo_x in _aqo_diag_row[_aqo_k])
                      for _aqo_k in ("old_full_correct", "new_full_correct")]
    if not (_aqo_diag_kept[0] and _aqo_diag_row.get("run_id") == _aqo_diag_prov["run_id"]
            and _aqo_diag_row.get("inputs") == list(FV_MASK[("antonym", _aqo_diag_s)]["inputs"])
            and _aqo_diag_row.get("retained_indexes")
            == [_aqo_i for _aqo_i, _aqo_keep in enumerate(ICL_MASK[("antonym", _aqo_diag_s)]["ft"]) if _aqo_keep]):
        _aqo_diag_status = "paired evaluation results unavailable"  # stale or incomplete rows outrank a missing arm
        break
    if not _aqo_diag_kept[1]:
        _aqo_diag_status = "query-only per-item correctness not retained"
_aqo_diag_conc, _aqo_diag_compare = None, None
if _aqo_diag_status != "computed":
    print(f"concordance: {_aqo_diag_status}")
else:
    print("concordance on retained rows: a = both correct, b = old correct and new wrong, "
          "c = old wrong and new correct, d = both wrong")
    _aqo_diag_conc, _aqo_diag_compare = {}, {}
    for _aqo_diag_s in _AQO_SEEDS:
        _aqo_diag_row = _aqo_diag_res["seeds"][_aqo_diag_s]
        _aqo_diag_items = [(bool(_aqo_diag_row["old_full_correct"][_aqo_i]),
                            bool(_aqo_diag_row["new_full_correct"][_aqo_i]))
                           for _aqo_i in _aqo_diag_row["retained_indexes"]]  # the evaluation's ordered retained rows
        _aqo_diag_c = {_aqo_k: sum(_aqo_it == _aqo_cell for _aqo_it in _aqo_diag_items) for _aqo_k, _aqo_cell in
                       (("a", (True, True)), ("b", (True, False)), ("c", (False, True)), ("d", (False, False)))}
        _aqo_diag_obs = (sum(_aqo_diag_c.values()), _aqo_diag_c["a"] + _aqo_diag_c["b"],
                         _aqo_diag_c["a"] + _aqo_diag_c["c"])  # (retained, old, new) rebuilt from the per-item rows
        for _aqo_diag_got, _aqo_diag_key, _aqo_diag_sum in zip(
                _aqo_diag_obs, ("retained_count", "old_correct", "new_correct"), ("a+b+c+d", "a+b", "a+c")):
            _aqo_require(_aqo_diag_got == _aqo_diag_row.get(_aqo_diag_key), "ANTONYM_QUERY_ONLY_RESULTS", _aqo_diag_s,
                         f"{_aqo_diag_sum}={_aqo_diag_got} differs from "
                         f"{_aqo_diag_key}={_aqo_diag_row.get(_aqo_diag_key)}")
        _aqo_diag_conc[_aqo_diag_s] = _aqo_diag_c
        _aqo_diag_compare[_aqo_diag_s] = {"expected": _AQO_DIAG_EXPECTED[_aqo_diag_s], "observed": _aqo_diag_obs,
                                          "match": _aqo_diag_obs == _AQO_DIAG_EXPECTED[_aqo_diag_s]}
        print(f"[antonym seed {_aqo_diag_s}] a={_aqo_diag_c['a']} b={_aqo_diag_c['b']} c={_aqo_diag_c['c']} "
              f"d={_aqo_diag_c['d']} | (retained, old, new) expected={_AQO_DIAG_EXPECTED[_aqo_diag_s]} "
              f"observed={_aqo_diag_obs} {'MATCH' if _aqo_diag_compare[_aqo_diag_s]['match'] else 'MISMATCH'}")
_aqo_diag_bad = [_aqo_s for _aqo_s, _aqo_r in (_aqo_diag_compare or {}).items() if not _aqo_r["match"]]
AQO_DIAGNOSTICS = {  # bound only after all four items are computed; a mismatch raises below, after binding
    "run_id": _aqo_diag_prov["run_id"], "antonym_cross_seed_cos": _aqo_diag_cross,
    "antonym_cross_seed_summary": _aqo_diag_summary, "antonym_norms": _aqo_diag_ratio,
    "present_past_seed0_cos": _aqo_diag_groups["seed 0 vs seeds 1-4"],
    "present_past_seeds_1_4_cos": _aqo_diag_groups["pairs among seeds 1-4"], "present_past_tensors": _aqo_diag_pp,
    "concordance_status": _aqo_diag_status, "concordance": _aqo_diag_conc,
    "expected_comparison": _aqo_diag_compare, "mismatched_seeds": _aqo_diag_bad}
_aqo_require(not _aqo_diag_bad, "aqo-diagnostics-v1", ",".join(map(str, _aqo_diag_bad)),
             "observed (retained, old, new) differ from expected; reproduction is not established "
             "(results kept in AQO_DIAGNOSTICS)")


[antonym seeds 0-1] cos(original, original) = 0.996148
[antonym seeds 0-2] cos(original, original) = 0.996379
[antonym seeds 0-3] cos(original, original) = 0.996066
[antonym seeds 0-4] cos(original, original) = 0.996903
[antonym seeds 1-2] cos(original, original) = 0.995738
[antonym seeds 1-3] cos(original, original) = 0.995306
[antonym seeds 1-4] cos(original, original) = 0.996607
[antonym seeds 2-3] cos(original, original) = 0.995637
[antonym seeds 2-4] cos(original, original) = 0.996043
[antonym seeds 3-4] cos(original, original) = 0.995459
antonym cross-seed cos over 10 pairs: min=0.995306 mean=0.996029 max=0.996903
[antonym seed 0] norm(original) = 42.223091 norm(query-only) = 42.642071 query-only/original = 1.009923
[antonym seed 1] norm(original) = 42.599468 norm(query-only) = 42.632153 query-only/original = 1.000767
[antonym seed 2] norm(original) = 42.289177 norm(query-only) = 42.748009 query-only/original = 1.010850
[antonym seed 3] norm(original) = 42.799992 norm(query-only)

In [22]:
# pp-swap-v1: present_past vector/item swap. Scores FV_TENSORS[("present_past", v)] on item seed i's 40 ordered TEST
# items through Cell 9's own fv_pred / exact_capital_match (layer 9, last position, additive). Defines no functions.
_pp_missing = [_pp_n for _pp_n in ("TASKS", "FV_TENSORS", "ICL_MASK", "fv_pred", "exact_capital_match",
                                   "_fv_generate", "first_token_id", "split_data") if _pp_n not in globals()]
if _pp_missing:
    raise RuntimeError(f"pp-swap-v1 preflight: {_pp_missing} undefined; run Cells 3, 8, 9, 11 and 12 first")
if "present_past" not in TASKS:
    raise RuntimeError("pp-swap-v1 preflight: TASKS has no present_past rows; run Cell 9 first")
_pp_ns = fv_pred.__globals__  # where fv_pred, and _fv_generate under exact_capital_match, look up fv_vector
if not (exact_capital_match.__globals__ is _pp_ns and _fv_generate.__globals__ is _pp_ns):
    raise RuntimeError("pp-swap-v1 preflight: fv_pred, exact_capital_match and _fv_generate do not share one "
                       "globals dict, so a single fv_vector binding would not reach both scorers")
_pp_vec = {_pp_s: FV_TENSORS.get(("present_past", _pp_s)) for _pp_s in (0, 1)}
for _pp_s, _pp_t in _pp_vec.items():
    if not (isinstance(_pp_t, torch.Tensor) and tuple(_pp_t.shape) == (1, 4096) and _pp_t.dtype == torch.float16
            and bool(torch.isfinite(_pp_t).all())):
        raise RuntimeError(f"pp-swap-v1 preflight [vector seed {_pp_s}]: FV_TENSORS[('present_past', {_pp_s})] "
                           f"must be a finite (1, 4096) torch.float16 tensor; got {type(_pp_t).__name__} "
                           f"shape={tuple(getattr(_pp_t, 'shape', ()))} dtype={getattr(_pp_t, 'dtype', None)}")
_pp_items, _pp_mask = {}, {}
for _pp_s in (0, 1):
    _pp_items[_pp_s] = split_data(TASKS["present_past"], seed=_pp_s)[1]  # Cell 9 line 35: the 40 ordered TEST items
    _pp_m = ICL_MASK.get(("present_past", _pp_s))
    _pp_at = f"pp-swap-v1 preflight [item seed {_pp_s}]: ICL_MASK[('present_past', {_pp_s})]"
    if not (isinstance(_pp_m, dict) and {"inputs", "golds", "ft", "exact"} <= set(_pp_m)):
        raise RuntimeError(f"{_pp_at} must be a dict with inputs, golds, ft and exact")
    if len(_pp_items[_pp_s]) != 40:
        raise RuntimeError(f"{_pp_at}: split_data gave {len(_pp_items[_pp_s])} test items, expected 40")
    for _pp_k, _pp_col in (("inputs", 0), ("golds", 1)):
        if _pp_m[_pp_k] != [_pp_row[_pp_col] for _pp_row in _pp_items[_pp_s]]:
            raise RuntimeError(f"{_pp_at}['{_pp_k}'] differ from split_data(TASKS['present_past'], "
                               f"seed={_pp_s})[1] in content or order")
    for _pp_k in ("ft", "exact"):
        if not (isinstance(_pp_m[_pp_k], list) and len(_pp_m[_pp_k]) == 40
                and all(isinstance(_pp_b, (bool, np.bool_)) for _pp_b in _pp_m[_pp_k])):
            raise RuntimeError(f"{_pp_at}['{_pp_k}'] must be a list of 40 bools")
    _pp_mask[_pp_s] = _pp_m
_pp_saved = {_pp_s: _pp_t.clone() for _pp_s, _pp_t in _pp_vec.items()}  # the post-check compares against these
# Score the two controls first, then the swaps. Rebinding fv_vector in the scorers' namespace is the one sanctioned
# mutation; finally restores the previous object (or its absence) by identity, as aqo-evaluate-v1's _aqo_score does.
_pp_absent, _pp_scored = object(), {}
_pp_prev = _pp_ns.get("fv_vector", _pp_absent)
try:
    with torch.no_grad():  # as Cell 9 scores
        for _pp_v, _pp_i in ((0, 0), (1, 1), (0, 1), (1, 0)):  # (vector seed, item seed)
            _pp_ns["fv_vector"] = _pp_vec[_pp_v]
            _pp_ft = [bool(fv_pred(_pp_x) == first_token_id(_pp_y)) for _pp_x, _pp_y in _pp_items[_pp_i]]  # line 40
            _pp_ex = [bool(exact_capital_match(_pp_x, _pp_y)) for _pp_x, _pp_y in _pp_items[_pp_i]]  # Cell 9 line 41
            _pp_scored[(_pp_v, _pp_i)] = (_pp_ft, _pp_ex)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
finally:
    if _pp_prev is _pp_absent:
        _pp_ns.pop("fv_vector", None)
    else:
        _pp_ns["fv_vector"] = _pp_prev
if _pp_ns.get("fv_vector", _pp_absent) is not _pp_prev:
    raise RuntimeError("pp-swap-v1: fv_vector was not restored to its previous binding; results not published")
for _pp_s, _pp_t in _pp_vec.items():
    if not (FV_TENSORS.get(("present_past", _pp_s)) is _pp_t and _pp_t.dtype == _pp_saved[_pp_s].dtype
            and torch.equal(_pp_t, _pp_saved[_pp_s])):
        raise RuntimeError(f"pp-swap-v1 [vector seed {_pp_s}]: FV_TENSORS[('present_past', {_pp_s})] changed "
                           "during scoring; results not published")
_pp_rows = {}
for (_pp_v, _pp_i), (_pp_ft, _pp_ex) in _pp_scored.items():
    _pp_m = _pp_mask[_pp_i]  # the ITEM seed's own ICL mask (Cell 13 _filtered): ft against ft, exact against exact
    _pp_rows[(_pp_v, _pp_i)] = {
        "vector_seed": _pp_v, "item_seed": _pp_i, "control": _pp_v == _pp_i, "n": len(_pp_ft), "ft": _pp_ft,
        "exact": _pp_ex, "ft_count": sum(_pp_ft), "exact_count": sum(_pp_ex),
        "filtered_ft": (sum(bool(_pp_a and _pp_b) for _pp_a, _pp_b in zip(_pp_m["ft"], _pp_ft)),
                        sum(map(bool, _pp_m["ft"]))),
        "filtered_exact": (sum(bool(_pp_a and _pp_b) for _pp_a, _pp_b in zip(_pp_m["exact"], _pp_ex)),
                           sum(map(bool, _pp_m["exact"])))}
# Print all four conditions and the 2x2 table, bind PP_SWAP, and only then check the controls.
for _pp_r in _pp_rows.values():
    print(f"[present_past vector {_pp_r['vector_seed']} on items {_pp_r['item_seed']}] "
          f"ft {_pp_r['ft_count']}/{_pp_r['n']} exact {_pp_r['exact_count']}/{_pp_r['n']} | "
          f"filtered ft {_pp_r['filtered_ft'][0]}/{_pp_r['filtered_ft'][1]} "
          f"exact {_pp_r['filtered_exact'][0]}/{_pp_r['filtered_exact'][1]}")
_pp_table = [[_pp_rows[(_pp_v, _pp_i)]["ft_count"] for _pp_i in (0, 1)] for _pp_v in (0, 1)]  # [vector][item]
print("present_past first-token counts out of 40 (rows = vector seed, columns = item seed):")
print(f"{'':8}  {'items 0':>7}  {'items 1':>7}")
for _pp_v, (_pp_c0, _pp_c1) in enumerate(_pp_table):
    print(f"vector {_pp_v}  {_pp_c0:>7}  {_pp_c1:>7}")
_pp_expected = {(0, 0): {"ft": 1, "exact": 1, "filtered_ft": (1, 38)},  # known values from the earlier
                (1, 1): {"ft": 10, "exact": 10, "filtered_ft": (10, 39)}}  # Cell 9 and Cell 13 runs
_pp_controls = {}
for _pp_key, _pp_want in _pp_expected.items():
    _pp_got = {"ft": _pp_rows[_pp_key]["ft_count"], "exact": _pp_rows[_pp_key]["exact_count"],
               "filtered_ft": _pp_rows[_pp_key]["filtered_ft"]}
    _pp_controls[_pp_key] = {"expected": _pp_want, "observed": _pp_got, "match": _pp_got == _pp_want}
_pp_failed = [_pp_key for _pp_key, _pp_c in _pp_controls.items() if not _pp_c["match"]]
PP_SWAP = {"conditions": _pp_rows, "ft_table": {"rows": "vector seed", "columns": "item seed", "counts": _pp_table},
           "controls": _pp_controls, "failed_controls": _pp_failed}
_pp_say = "ft {ft}/40, exact {exact}/40, filtered ft {filtered_ft[0]}/{filtered_ft[1]}"
if _pp_failed:
    raise RuntimeError("pp-swap-v1: control failed to reproduce: " + "; ".join(
        f"vector {_pp_v} on items {_pp_i} expected " + _pp_say.format(**_pp_controls[(_pp_v, _pp_i)]["expected"])
        + " but observed " + _pp_say.format(**_pp_controls[(_pp_v, _pp_i)]["observed"])
        for _pp_v, _pp_i in _pp_failed) + " (all four conditions are printed above and kept in PP_SWAP)")
print("controls reproduced: vector 0 on items 0 and vector 1 on items 1 match ft, exact and filtered ft")


[present_past vector 0 on items 0] ft 1/40 exact 1/40 | filtered ft 1/38 exact 1/38
[present_past vector 1 on items 1] ft 10/40 exact 10/40 | filtered ft 10/39 exact 10/39
[present_past vector 0 on items 1] ft 6/40 exact 6/40 | filtered ft 6/39 exact 6/39
[present_past vector 1 on items 0] ft 2/40 exact 2/40 | filtered ft 2/38 exact 2/38
present_past first-token counts out of 40 (rows = vector seed, columns = item seed):
          items 0  items 1
vector 0        1        6
vector 1        2       10
controls reproduced: vector 0 on items 0 and vector 1 on items 1 match ft, exact and filtered ft
